# Caravela — Projeto Parte I
## Engenharia Gramatical
## MEI/2026-27
#### Grupo: Nome Apelido (aXXXXX), Nome Apelido (aXXXXX), Nome Apelido (aXXXXX)

Este notebook é o relatório da Parte I do projeto. Está organizado pelas quatro tarefas do enunciado e o código de cada secção corre de cima para baixo, por isso os resultados que aparecem no texto são os que as células imprimem.

1. Desenho da linguagem
2. Especificação léxica e sintática
3. Especificação semântica
4. Qualidade da linguagem e da gramática
5. Conclusão

Anexo A (ficheiros originais das aulas) e Anexo B (como o compilador gera os shaders).

## 1. Desenho da linguagem

### 1.1 Contextualização

No âmbito da unidade curricular de Engenharia Gramatical, foi-nos proposto desenhar e especificar uma linguagem para um domínio concreto. Escolhemos um problema que conhecemos bem da unidade curricular de Visualização em Tempo Real (VTR), onde os trabalhos práticos são feitos no Nau, um motor gráfico desenvolvido na Universidade do Minho.

Um motor gráfico é, basicamente, o programa que recebe a descrição de uma cena 3D e a desenha no ecrã usando a placa gráfica. Essa descrição diz que objetos existem, onde estão, de onde estamos a olhar para eles e de onde vem a luz.

No Nau não se programa o motor em si. Cada trabalho é um conjunto de ficheiros de texto que o motor lê e interpreta:

* projeto (.xml): é o ficheiro principal. Descreve as cenas (os objetos 3D), a câmara, as luzes e a ordem pela qual tudo é desenhado.

* biblioteca de materiais (.mlib): define o aspeto de cada objeto (a cor e o brilho) e qual o programa que o desenha.

* shaders (.vert e .frag): pequenos programas, escritos em GLSL (uma linguagem parecida com C), que correm na placa gráfica e calculam o aspeto final do objeto.

Os shaders vêm sempre aos pares, porque a placa gráfica desenha um objeto em duas fases:

* vertex shader (.vert): corre uma vez para cada vértice do objeto, ou seja, cada canto dos triângulos que formam o modelo 3D. Calcula onde é que esse vértice aparece no ecrã, tendo em conta a posição da câmara.
* fragment shader (.frag): corre uma vez para cada píxel que o objeto ocupa no ecrã, e calcula a cor final desse píxel.

Há um pormenor importante para perceber o resto do relatório. Os shaders precisam de receber valores de fora, como a cor do objeto, a direção da luz ou a posição da câmara. A estas entradas chama-se uniforms, e cada uma tem um nome dentro do shader. Por exemplo, a direção da luz pode chamar-se l_dir.

É no ficheiro .mlib que se diz ao Nau que valor deve pôr em cada uniform, e a ligação é feita pelo nome. Ou seja, o nome escrito no .mlib tem de ser exatamente igual ao que está no shader. Se não for, o Nau não avisa: o valor fica simplesmente a zero. Este é um dos problemas que a Caravela resolve, como se explica a seguir.

### 1.2 O problema

O projeto mais simples da aula 2 de VTR desenha uma chaleira, vista por uma câmara e iluminada por uma luz. Parece pouca coisa, mas para isto são precisas 175 linhas, espalhadas por quatro ficheiros: 67 no projeto (.xml), 73 na biblioteca de materiais (.mlib) e 35 nos dois shaders (25 no .vert e 10 no .frag).

Para se ter uma ideia do tipo de texto que é preciso escrever, aqui fica um pedaço do projeto:

```xml
<camera name="MainCamera">
    <viewport name="MainViewport" />
    <POSITION x="1.46" y="1.05" z="1.75" />
    <LOOK_AT_POINT x="0" y="0" z="0" />
</camera>
...
<valueof uniform="l_dir" type="LIGHT" component="DIRECTION" id="0" />
```

Escrever estes ficheiros à mão tem alguns problemas:

1. **Há muita repetição.** Linhas como a do valueof repetem-se em quase todos os materiais. Os próprios shaders também são quase sempre os mesmos: de projeto para projeto copiam-se os .vert e .frag e mudam-se só os nomes dos uniforms.

2. **Tudo se liga por nomes, em ficheiros diferentes.** A câmara refere o viewport pelo nome, o passe refere a câmara, as cenas e as luzes pelo nome, e o .mlib refere os uniforms do shader pelo nome. Ninguém verifica se estes nomes batem certo.

3. **Quando há um erro, o Nau não avisa.** Se um nome estiver mal escrito, o valor fica a zero e o objeto aparece preto, ou nem aparece. Para encontrar o erro temos de comparar os ficheiros linha a linha.

Do ponto de vista da Engenharia Gramatical, o problema é que o XML só verifica a forma do texto (se as etiquetas estão bem abertas e fechadas), mas não o significado (se aquilo a que um nome se refere existe e é do tipo certo).

### 1.3 Objetivo da linguagem

A **Caravela** é uma linguagem de domínio específico, pensada para descrever cenas 3D do Nau de forma curta e fácil de ler. Em vez de escrever os ficheiros do Nau à mão, escreve-se um programa em Caravela, e um compilador gera a partir dele todos os ficheiros de texto de que o Nau precisa: o projeto (.xml), a biblioteca de materiais (.mlib) e os shaders (.vert e .frag).

Os modelos 3D (.obj) e as imagens das texturas continuam a ser ficheiros à parte, e no programa só se indica o caminho para eles. O Nau em si não é alterado.

A escolha do nome está relacionada com o próprio motor. Tal como a caravela era uma embarcação mais pequena e mais fácil de manobrar do que a nau, a linguagem pretende ser uma forma mais simples e acessível de trabalhar com o Nau.

Uma das ideias principais da Caravela é que o utilizador não escreve shaders. Em cada material diz-se apenas que efeito se quer obter: por exemplo, lighting phong para iluminação com brilho, ou uma camada de textura para o lado da noite. A partir daí, é o compilador que gera o código GLSL, usando modelos já preparados, e que trata também das ligações entre o shader e o motor.

Como é o compilador que escreve os dois lados dessas ligações (o shader e o .mlib), os nomes coincidem sempre, e o utilizador nem precisa de saber que os uniforms existem.

Com esta linguagem temos três objetivos principais:

1. **Escrever menos**. Cada elemento da cena (textura, material, câmara, luz, cena) é declarado uma só vez, num bloco com nome. Tudo o que é repetitivo, incluindo os shaders, passa a ser gerado automaticamente.
2. **Detetar os erros antes de chegar ao Nau.** Como todos os nomes estão declarados no mesmo programa, conseguimos verificar se cada referência existe e se é do tipo certo. Por exemplo, um passe só pode desenhar cenas, e não câmaras. Também conseguimos verificar se as opções de um material fazem sentido juntas: por exemplo, o brilho só existe na iluminação phong.
3. **Permitir cenas parametrizáveis**. Com variáveis, ciclos, condições e templates é possível gerar muitos objetos, ou várias versões da mesma cena, com poucas linhas.

**Âmbito**.
A Caravela cobre o que é dado nas aulas 2 (iluminação) e 3 (texturas) de VTR, e os efeitos disponíveis são exatamente os dessas aulas. Funcionalidades mais avançadas do Nau, como sombras, vários passes de desenho encadeados ou shaders escritos pelo próprio utilizador, ficam de fora nesta versão.

**Utilizadores**.
 A linguagem é pensada sobretudo para os alunos de VTR e, de forma geral, para quem quer montar no Nau as cenas das aulas, ou variações delas, sem ter de escrever XML nem GLSL. Ou seja, pessoas que sabem o que querem ver no ecrã (que objetos, com que cores, texturas e iluminação), mas que não querem perder tempo com a configuração.

### 1.4 Conceitos principais

A tabela seguinte resume os principais conceitos da Caravela, o que cada um representa e como se escreve na linguagem.

| Conceito | Descrição | Em Caravela |
|---|---|---|
| Textura | imagem aplicada à superfície de um objeto (por exemplo, o mapa da Terra numa esfera) | `texture earth = "earthDay.jpg" mipmap;` |
| Material | aspeto de um objeto: a cor, a iluminação e as camadas de textura | `material amarelo { ... }` |
| Iluminação | forma como a luz é calculada sobre o objeto; o compilador gera o shader correspondente | `lighting per_vertex;`, `lighting phong { shininess 100; }`, `lighting toon { levels 4; }`, `lighting none;` |
| Camada | textura com uma função definida no material: cor de dia, zonas que brilham, luzes de noite ou nuvens | `layer day earth;`, `layer specular spec;`, `layer night eNight;`, `layer clouds noise speed 0.0001;` |
| Cena | conjunto de objetos 3D, carregados de um ficheiro ou gerados pela linguagem (esfera, caixa, grelha) | `scene mundo { ... }` |
| Objeto e transformações | forma geométrica, com a sua posição, tamanho e rotação | `box caixa : azul { translate (2, 0, 0); }` |
| Câmara | ponto de onde se observa a cena e direção para onde se olha | `camera cam { ... }` |
| Luz | direção de onde vem a iluminação | `light sol directional { ... }` |
| Passe | instrução de desenho: que cenas desenhar, com que câmara e com que luzes | `pass render { ... }` |
| Variável | valor com nome, que pode ser usado em expressões | `let raio = 4;` |
| Ciclo e condição | repetir objetos ou escolher entre alternativas | `for i in 0..n { ... }`, `if ... { } else { }` |
| Template | grupo de objetos com parâmetros, que pode ser reutilizado | `template pilar(h: num, mat: material) { ... }` |

Os quatro modos de iluminação correspondem aos shaders usados nas aulas:

- `per_vertex` calcula a luz apenas nos vértices dos triângulos (é o shader `simple` da aula 2);
- `phong` calcula a luz em cada píxel e acrescenta o brilho especular (é o shader `v4`, também da aula 2);
- `toon` arredonda a intensidade da luz a um número fixo de patamares, o que dá um aspeto de desenho animado;
- `none` mostra apenas a cor do objeto, sem luz.

### 1.5 Como as construções servem o objetivo

**Declarações com nome.** Texturas, materiais, cenas, câmaras e luzes são declarados com um nome e depois usados através dele. Continua a ser uma ligação por nomes, como no XML, mas agora está tudo no mesmo programa. Assim, o compilador consegue construir uma tabela de símbolos e verificar cada referência.

**Efeitos em vez de shaders.** O material diz o resultado que se quer (lighting, layer) e não como se calcula. É o compilador que escolhe o shader, gera os .vert e .frag e escreve no .mlib os uniforms e as unidades de textura. Os shaders são montados a partir de pedaços de código já preparados, retirados dos próprios shaders das aulas 2 e 3, e é gerado um shader por cada combinação de modo e camadas, e não um por material. Valores como a cor, o brilho ou a direção da luz não ficam no shader: vão para o .mlib, como já acontece nas aulas. Com isto desaparecem dois tipos de erros silenciosos: nomes de uniforms que não coincidem e unidades de textura trocadas. Em troca, há combinações que deixam de ser válidas (por exemplo, specular só com phong), e o compilador verifica-as. Os detalhes de como cada construção é traduzida estão no Anexo B.

**Blocos aninhados.** Materiais, cenas e câmaras são blocos `{ ... }` com as suas propriedades. Uma cena tem objetos, e cada objeto tem as suas transformações, por isso o programa tem a mesma estrutura em árvore que a própria cena.

**Variáveis, ciclos, condições e templates.** O XML do Nau só aceita valores fixos. Na Caravela, estas construções são resolvidas pelo compilador antes de gerar os ficheiros. Por exemplo, um ciclo que cria oito pilares dá origem a oito elementos `<geometry>` no XML, cada um já com a sua posição. O `if` serve, por exemplo, para trocar de modo de iluminação mudando só uma variável, ou para alternar materiais dentro de um ciclo.

**Relação com os requisitos mínimos do enunciado:**

| Requisito | Onde aparece na Caravela |
|---|---|
| Variáveis e declarações | `let` e as declarações de `texture`, `material`, `scene`, `camera`, `light`, `pass` e `template` |
| Âmbitos | global, corpo do `for`, ramos do `if` e corpo do `template` |
| Expressões e valores | contas e comparações em posições, ângulos, cores, parâmetros de iluminação e condições |
| Construções compostas e aninhadas | cena → objetos → transformações; material → iluminação e camadas; `for` e `if` dentro de cenas e `if` dentro de passes |
| Ramificação e repetição | `if`/`else` e `for` |
| Propriedades que a sintaxe não determina | ver lista abaixo |

**Propriedades semânticas** (não podem ser verificadas pela gramática independente de contexto):

- **Nomes:** todos os nomes usados estão declarados e são visíveis no âmbito onde são usados, e não há duas declarações com o mesmo nome no mesmo âmbito. Os nomes de cores (`white`, `black`, `red`, `green`, `blue`, `yellow`, `grey`) não são palavras reservadas: são constantes do tipo `color` que já existem no ambiente inicial, e por isso não podem ser declaradas de novo (`let red = 1;` é um erro).
- **Ordem das declarações:** os nomes globais podem ser usados antes de serem declarados, mas os `let` globais não podem depender uns dos outros em ciclo (`let a = b + 1; let b = a;` é um erro). Os `let` locais só podem ser usados depois de declarados.
- **Tipos das referências:** `draw` só aceita cenas, `camera` só aceita câmaras, `layer` só aceita texturas, e os argumentos de um `template` têm de ter o tipo declarado nos parâmetros (`draw cam` é um erro, porque `cam` é uma câmara).
- **Valores válidos:** `near > 0`, `far > near`, `shininess > 0`, `levels >= 2` e as componentes das cores entre 0 e 1.
- **Coerência do material:**
  - `shininess` só pode aparecer em `lighting phong`, e `levels` só em `lighting toon`;
  - a camada `specular` exige `lighting phong`, e a camada `night` exige um modo com luz (não `none`);
  - cada tipo de camada aparece no máximo uma vez por material, e `specular`, `night` e `clouds` exigem uma camada `day`;
  - um material tem no máximo um `lighting` (por omissão, `per_vertex`).
- **Templates:** um template não se pode chamar a si próprio, direta ou indiretamente.
- **Unidades de textura:** cada camada de um material recebe um número de unidade diferente (0, 1, 2, ...), pela ordem em que aparece. Este valor não é verificado, mas sim **calculado** pelo compilador.

### 1.6 Exemplos de programas

Como o Nau é um motor muito grande, com muitas funcionalidades, não fazia sentido tentar cobri-lo todo nesta linguagem. Por isso, limitámos a Caravela às funcionalidades dadas nas aulas 2 e 3 de VTR, que são a iluminação e as texturas. Dentro destes limites, a linguagem permite descrever qualquer cena: com os modelos que se quiser, objetos gerados, câmaras, luzes, ciclos e templates.

Para a pôr à prova, reescrevemos em Caravela três projetos dessas aulas (exemplos 1, 2 e 4). Cada um está completo, ou seja, produz o mesmo resultado que todos os ficheiros originais juntos. Os ficheiros originais estão no Anexo A, para comparar. O exemplo 3 é uma cena nova, que não vem das aulas, e mostra o que a Caravela acrescenta ao Nau.

| Exemplo | Original (aulas de VTR) | Caravela |
|---|---|---|
| 1. Iluminação simples (aula 2) | 175 linhas em 4 ficheiros (`.xml`, `.mlib`, `.vert`, `.frag`) | 28 linhas, 1 ficheiro |
| 2. Iluminação por píxel (aula 2) | 191 linhas em 4 ficheiros (o `.mlib` é o mesmo do exemplo 1) | 34 linhas, 1 ficheiro, com os três modos |
| 3. Templo com pilares (cena nova) | — (não existe nas aulas) | 38 linhas, 1 ficheiro |
| 4. Texturas da Terra (aula 3) | 232 linhas em 4 ficheiros, mais a biblioteca de ruído `noise.frag` (132 linhas) | 33 linhas, 1 ficheiro |

**Exemplo 1: o projeto da aula 2 (`simple.xml`).** Uma chaleira vista pela câmara e iluminada pelo sol, com a luz calculada nos vértices.

```
project "Simple";

scene teapot_l = model "models/teapot.3ds" unitize swap_yz;
scene teapot_s = model "models/teapot.obj" unitize;
scene al       = model "models/al.obj"     unitize;
scene plane    = grid length 20 divisions 2;

material Yellow { diffuse rgb(1.0, 0.87, 0.0); }
material Blue   { diffuse rgb(0.0, 0.34, 0.72); }

camera MainCamera {
    position (1.46, 1.05, 1.75);
    look_at  (0, 0, 0);
    near     0.01;
}

light Sun directional {
    direction (2, 1, 2);
}

background white;

pass pass1 {
    draw   teapot_s;
    camera MainCamera;
    lights Sun;
    lighting per_vertex for all;   // ilumina tudo desta forma, mantendo as cores do modelo
}
```

Ao traduzir este projeto, reparámos em duas coisas no original:

- O projeto declara quatro cenas (`teapot_l`, `teapot_s`, `al` e `plane`), mas só desenha a `teapot_s`. O `.mlib` também declara os materiais `Yellow` e `Blue`, que não são usados por nenhum objeto. Mantivemos estas declarações para o programa ser equivalente ao original, mas o compilador poderia avisar que não são usadas.
- No original, o `.xml` escolhe o shader com `fromLibrary="simple" fromMaterial="simple"`, dois nomes que têm de bater certo com o `.mlib`. Por sua vez, o `.mlib` refere os uniforms `m_pvm`, `m_view`, `m_normal`, `l_dir` e `diffuse`, que têm de bater certo com o `.vert`. Na Caravela estas ligações deixam de existir: basta escrever `lighting per_vertex`, e os shaders e as ligações são gerados juntos.

Os ficheiros originais deste projeto (`simple.xml`, `simple.mlib`, `simple.vert` e `simple.frag`) estão no Anexo A.1.

**Exemplo 2: o projeto da aula 2 com iluminação por píxel (`simple4.xml`).** Nesta aula comparam-se várias formas de iluminar o mesmo objeto. No original, para mudar de modo é preciso editar o `fromMaterial="v4"` no `.xml`. Além disso, cada modo tem os seus próprios shaders, com nomes de uniforms diferentes: `m_v`, `m_n` e `ld` no `v4`, mas `m_view`, `m_normal` e `l_dir` no `simple`. Na Caravela basta mudar o valor da variável `modo`, e o compilador só gera o shader do modo escolhido.

```
project "Simple";

let modo = 2;          // 1 = por vértice (simple), 2 = phong (v4), 3 = toon

scene rb         = model "models/rainbowBench.obj" unitize;
scene al         = model "models/al.obj"           unitize;
scene teapot_obj = model "models/teapot.obj"       unitize;
scene teapot_3ds = model "models/teapot.3ds"       unitize swap_yz;
scene plane      = grid length 20 divisions 2;

camera MainCamera {
    position (1.46, 1.05, 1.75);
    look_at  (0, 0, 0);
    near     0.1;
}

light Sun directional {
    direction (-2, -1, -2);
}

background rgb(0.15, 0.25, 0.45);

pass pass1 {
    draw   teapot_3ds;
    camera MainCamera;
    lights Sun;
    if modo == 2 {
        lighting phong { shininess 100; } for all;
    } else if modo == 3 {
        lighting toon { levels 4; } for all;
    } else {
        lighting per_vertex for all;
    }
}
```

O `.mlib` deste projeto é o mesmo do exemplo 1 (`simple.mlib`), onde já está o material `v4` com `shininess` 100.

Os ficheiros originais deste projeto (`simple4.xml`, `v4.vert` e `v4.frag`) estão no Anexo A.2.

**Exemplo 3: uma cena gerada com template, ciclo e condição.** Este exemplo não vem das aulas. Serve para mostrar o que a Caravela permite fazer e o Nau não. São oito pilares em círculo, que alternam entre duas alturas e dois materiais. O compilador calcula o ângulo e a posição de cada pilar e gera oito objetos no XML, cada um com um nome diferente. À mão, seria preciso escrever oito elementos `<geometry>`, com as posições e rotações calculadas à parte.

```
let n    = 8;
let raio = 4;

material azul   { diffuse rgb(0.0, 0.34, 0.72); lighting phong { shininess 60; } }
material branco { diffuse white;                lighting per_vertex; }

template pilar(h: num, mat: material) {
    box p : mat {
        scale     (0.5, h, 0.5);
        translate (0, h / 2, 0);
    }
}

scene templo {
    for i in 0..n {                 // i = 0, 1, ..., n-1
        let ang = i * 360 / n;
        if i % 2 == 0 {
            pilar(3, azul)   { rotate ang around (0, 1, 0); translate (raio, 0, 0); }
        } else {
            pilar(2, branco) { rotate ang around (0, 1, 0); translate (raio, 0, 0); }
        }
    }
}

scene chao = grid length 20 divisions 2;

camera cam {
    position (8, 6, 8);
    look_at  (0, 1, 0);
}

light sol directional { direction (2, 1, 2); }

pass render {
    draw   templo, chao;
    camera cam;
    lights sol;
}
```

Este exemplo também mostra vários erros que o XML do Nau nunca detetaria, mas a Caravela deteta antes de gerar os ficheiros:

- escrever `pilar(3, azl)`, um material que não existe;
- trocar a ordem dos argumentos (`pilar(azul, 3)`);
- usar `ang` fora do ciclo;
- escrever `draw templo, cam`, quando `cam` não é uma cena;
- pôr `shininess` no material `branco`, que não usa `phong`.

Estas verificações são explicadas na secção 3.

**Exemplo 4: o projeto de texturas da aula 3 (`textureModel.xml`).** Uma esfera com o aspeto da Terra: a superfície durante o dia, as zonas que refletem a luz (o mar), as luzes das cidades no lado da noite e nuvens geradas por ruído, que se vão mexendo com o tempo.

No Nau, cada imagem é ligada ao shader através de um número de "unidade de textura", que tem de ser escrito em dois sítios diferentes do `.mlib`. Basta trocar um destes números para a esfera aparecer com a imagem errada, sem nenhum aviso. Na Caravela, o material só diz qual é o papel de cada imagem (`layer night eNight;`). É o compilador que atribui os números e gera o shader que junta as imagens todas.

```
project "Texture Model";

texture earth   = "otherTextures/earthDay.jpg"    mipmap;
texture spec    = "otherTextures/earthSpec.jpg"   mipmap;
texture eNight  = "otherTextures/earthLights.jpg" mipmap;
texture eClouds = "otherTextures/earthClouds.jpg" mipmap;   // declarada no original, mas nunca usada

material textured {
    lighting phong { shininess 128; }
    layer day      earth;
    layer specular spec;                  // só brilha onde a imagem é clara (o mar)
    layer night    eNight;                // aparece onde não chega luz
    layer clouds   noise speed 0.0001;    // nuvens procedimentais, animadas com o tempo
}

scene Sphere = model "models/sphere.obj" unitize;

camera MainCamera {
    position  (0, 0, 3);
    direction (0, 0, -1);                 // direção para onde olha, em vez de um ponto
    near      0.1;
}

light Sun directional { direction (2, -1, -2); }

background black;

pass render {
    draw   Sphere;
    camera MainCamera;
    lights Sun;
    material textured for all;            // substitui os materiais do modelo por este
}
```

Também neste projeto encontrámos algumas coisas no original:

- A textura `eClouds` é declarada e ligada à unidade 3, mas o shader não a usa. A linha que a lê está comentada, e as nuvens são geradas por ruído. Na Caravela, a textura fica declarada mas nenhuma camada a usa, por isso o compilador poderia avisar.
- O `.xml` refere `models/Sphere.obj`, mas o ficheiro chama-se `sphere.obj`. Em Windows isto funciona, mas em Linux o Nau não encontra o modelo. O compilador da Caravela poderia verificar se os ficheiros referidos existem.
- O `shininess` não tem nenhum `<valueof>` no `.mlib`. O valor 128 vem de um valor por omissão escrito dentro do próprio `texture.frag`. Na Caravela, o valor fica à vista, no material.

Os ficheiros originais deste projeto (`textureModel.xml`, `textureModel.mlib`, `texture.vert` e `texture.frag`) estão no Anexo A.3.

A construção `layer` tem sempre a mesma forma: `layer <papel> <origem>`. A origem é uma textura declarada ou, no caso das nuvens, `noise`, que é ruído gerado no próprio shader. Nos passes há duas formas de mudar o aspeto dos objetos. Com `lighting X for all` (exemplos 1 e 2), mantêm-se as cores do modelo e só muda a forma de iluminar. Com `material X for all` (exemplo 4), substitui-se o material inteiro.

## 2. Especificação léxica e sintática

Nesta secção especificamos a Caravela em duas partes, como nas aulas: primeiro o léxico (que "palavras" existem na linguagem) e depois a sintaxe (como é que essas palavras se podem juntar). Escrevemos tudo em Lark, por isso a gramática é ao mesmo tempo a especificação e um *parser* que conseguimos testar.

Por exemplo, em

```caravela
material azul { diffuse rgb(0.0, 0.34, 0.72); }
```

o analisador léxico parte o texto em tokens: palavras reservadas (`material`, `diffuse`, `rgb`), um identificador (`azul`), números (`0.0`, `0.34`, `0.72`) e pontuação (`{`, `(`, `,`, `;`, ...). O *lexer* não vê se os tokens estão pela ordem certa (isso é com o *parser*) nem se `azul` existe (isso é com a análise semântica, na secção 3).

### 2.1 Estrutura léxica

Os terminais da Caravela dividem-se em identificadores, números, texto, operadores, pontuação e comentários:

| Categoria | Expressão regular | Exemplos | Observações |
|---|---|---|---|
| Identificador (`ID`) | `[a-zA-Z_][a-zA-Z0-9_]*` | `teapot_s`, `MainCamera`, `raio` | Pode começar por `_`. Não pode ser uma palavra reservada. |
| Número (`NUMBER`) | `[0-9]+(\.[0-9]+)?` | `8`, `0.01`, `1.46` | Inteiros e decimais, sem sinal. O `-` é tratado na sintaxe (operador `SUB`). |
| Texto (`STRING`) | `"[^"\n]*"` | `"models/teapot.obj"` | Usado nos caminhos dos modelos e das texturas. |
| Comparação (`CMP_OP`) | `== \| != \| <= \| >= \| < \| >` | `i % 2 == 0` | Só aparece nas condições do `if`. |
| Soma e subtração | `+` (`ADD`), `-` (`SUB`) | `b + 1`, `-1` | O `-` serve para subtrair e para negar. |
| Multiplicação (`MUL_OP`) | `* \| / \| %` | `i * 45`, `i % 2` | Multiplicação, divisão e resto. |
| Intervalo (`_RANGE`) | `..` | `0..n` | Usado no `for`. Como o nome começa por `_`, o Lark não o põe na árvore. |
| Pontuação | `; , : = ( ) { }` | `(1, 0, 2)` | Blocos, vetores e listas. |
| Comentários (`COMMENT`) | `//[^\n]* \| /\*(.\|\n)*?\*/` | `// nota`, `/* bloco */` | São ignorados. O `*?` faz com que o comentário de bloco acabe no primeiro `*/`. |

Os espaços, tabs e mudanças de linha são ignorados com `%ignore /[ \t\r\n]+/`, e os comentários com `%ignore COMMENT`.

### 2.2 Palavras reservadas

A Caravela tem 58 palavras reservadas (o número é calculado na secção 4.1). Agrupámo-las assim:

* **Declarações e controlo:** `project`, `let`, `texture`, `material`, `scene`, `template`, `camera`, `light`, `background`, `pass`, `if`, `else`, `for`, `in`.
* **Materiais e shaders:** `diffuse`, `lighting`, `none`, `per_vertex`, `phong`, `toon`, `shininess`, `levels`, `layer`, `day`, `specular`, `night`, `clouds`, `noise`, `speed`, `mipmap`.
* **Cenas e geometria:** `model`, `grid`, `length`, `divisions`, `unitize`, `swap_yz`, `box`, `sphere`, `square`, `axis`, `translate`, `scale`, `rotate`, `around`.
* **Câmara, luz e passes:** `position`, `look_at`, `direction`, `near`, `far`, `fov`, `directional`, `color`, `draw`, `lights`, `all`.
* **Tipos e valores:** `num`, `vec3`, `rgb`, e também `color` e `material`, que já estão acima.

No léxico tivemos de tomar algumas decisões:

1. **Palavras reservadas e identificadores.** As palavras reservadas também encaixam na expressão regular do `ID`. Com `lexer="basic"`, o Lark lê primeiro a palavra inteira e só depois vê se é uma palavra reservada. Por isso `material` é sempre a palavra reservada, mas `materialx` ou `drawcam` são identificadores normais.
2. **As cores não são palavras reservadas.** `white`, `black`, `red`, ... são lidas como `ID`, e é a análise semântica que sabe que são cores, porque estão no ambiente inicial (`PRE`). Assim, para acrescentar uma cor não é preciso mexer no léxico nem na gramática.
3. **O `..` do `for`.** Em `0..n` podia haver confusão com um número decimal, mas não há: o `NUMBER` obriga a ter dígitos depois do ponto, por isso o `0` é lido como número e o `..` como `_RANGE`.
4. **Números negativos.** `-2` dá dois tokens, `SUB` e `NUMBER`. O sinal fica para o *parser* (`factor: SUB factor`), e assim `a - 2` e `a -2` são lidos da mesma maneira.

In [1]:
!pip -q install "lark[interegular]"

### 2.3 Gramática

A gramática completa está na célula seguinte. Usamos `parser="lalr"` e `lexer="basic"`, como nas aulas, e ainda `strict=True`, para o Lark dar erro se houver algum conflito. A gramática está dividida pelos mesmos grupos da secção 1.4: programa, texturas e materiais, cenas e templates, câmara e luz, passes, valores e, no fim, o léxico.

In [2]:
from lark import Lark

grammar = r"""
// ---------- Programa ----------
start: project? decl*

project: "project" STRING ";"

decl: let_decl
    | texture_decl
    | material_decl
    | scene_decl
    | template_decl
    | camera_decl
    | light_decl
    | background_decl
    | pass_decl

let_decl: "let" ID "=" value ";"

// ---------- Texturas e materiais ----------
texture_decl: "texture" ID "=" STRING MIPMAP? ";"

material_decl: "material" ID "{" mat_item* "}"

mat_item: diffuse
        | mat_lighting
        | layer

diffuse: "diffuse" color_val ";"

mat_lighting: "lighting" lighting_mode (lighting_params | ";")

lighting_mode: NONE
             | PER_VERTEX
             | PHONG
             | TOON

lighting_params: "{" lighting_param* "}"

lighting_param: shininess
              | levels

shininess: "shininess" num ";"

levels: "levels" num ";"

layer: "layer" layer_role layer_source ";"

layer_role: DAY
          | SPECULAR
          | NIGHT
          | CLOUDS

layer_source: ID
            | noise

noise: "noise" ("speed" num)?

// ---------- Cenas e templates ----------
scene_decl: "scene" ID "=" scene_source ";"
          | "scene" ID "{" scene_item* "}"

scene_source: model
            | grid

model: "model" STRING model_option*

model_option: UNITIZE
            | SWAP_YZ

grid: "grid" "length" num "divisions" num

scene_item: object
          | template_call
          | let_decl
          | scene_for
          | scene_if

object: shape ID (":" ID)? object_body

shape: BOX
     | SPHERE
     | SQUARE
     | AXIS

template_call: ID "(" args? ")" object_body

object_body: "{" transform* "}"
           | ";"

transform: translate
         | scale
         | rotate

translate: "translate" vec3_val ";"

scale: "scale" vec3_val ";"

rotate: "rotate" num "around" vec3_val ";"

scene_for: "for" ID "in" num _RANGE num "{" scene_item* "}"

scene_if: "if" cond scene_block ("else" (scene_block | scene_if))?

scene_block: "{" scene_item* "}"

template_decl: "template" ID "(" params? ")" "{" scene_item* "}"

params: param
      | params "," param

param: ID ":" type

type: NUM
    | VEC3
    | COLOR
    | MATERIAL

args: value
    | args "," value

// ---------- Câmara, luz e fundo ----------
camera_decl: "camera" ID "{" camera_prop* "}"

camera_prop: position
           | look_at
           | direction
           | near
           | far
           | fov

position: "position" vec3_val ";"

look_at: "look_at" vec3_val ";"

direction: "direction" vec3_val ";"

near: "near" num ";"

far: "far" num ";"

fov: "fov" num ";"

light_decl: "light" ID "directional" "{" light_prop* "}"

light_prop: direction
          | light_color

light_color: "color" color_val ";"

background_decl: "background" color_val ";"

// ---------- Passes ----------
pass_decl: "pass" ID "{" pass_item* "}"

pass_item: draw
         | pass_camera
         | pass_lights
         | pass_lighting
         | pass_material
         | pass_if

draw: "draw" id_list ";"

pass_camera: "camera" ID ";"

pass_lights: "lights" id_list ";"

pass_lighting: "lighting" lighting_mode lighting_params? "for" "all" ";"

pass_material: "material" ID "for" "all" ";"

pass_if: "if" cond pass_block ("else" (pass_block | pass_if))?

pass_block: "{" pass_item* "}"

id_list: ID
       | id_list "," ID

// ---------- Valores ----------
// Cada propriedade diz que tipo de valor aceita:
//   num    números e contas      shininess 100;   near 0.1;   rotate i * 45 ...
//   vec3   vetores (x, y, z)     position (1, 2, 3);
//   color  cores                 diffuse rgb(1, 0.87, 0);   background white;
//   cond   condições             if i % 2 == 0 { ... }

// números: como expr / term / factor nas aulas
num: num ADD term
   | num SUB term
   | term

term: term MUL_OP factor
    | factor

factor: NUMBER
      | ID
      | SUB factor
      | "(" num ")"

vec3: "(" num "," num "," num ")"

color: "rgb" vec3

cond: num CMP_OP num

// numa propriedade, em vez do valor pode vir uma variável
// (ou uma cor pré-definida: white, red, ... são ID)
vec3_val: vec3
        | ID

color_val: color
         | ID

// let e argumentos de templates aceitam qualquer tipo
value: num
     | vec3
     | color

// ---------- Léxico ----------
// operadores
CMP_OP: "==" | "!=" | "<=" | ">=" | "<" | ">"
ADD: "+"
SUB: "-"
MUL_OP: "*" | "/" | "%"
_RANGE: ".."

// palavras reservadas cujo valor fica na árvore
NONE: "none"
PER_VERTEX: "per_vertex"
PHONG: "phong"
TOON: "toon"
DAY: "day"
SPECULAR: "specular"
NIGHT: "night"
CLOUDS: "clouds"
UNITIZE: "unitize"
SWAP_YZ: "swap_yz"
MIPMAP: "mipmap"
BOX: "box"
SPHERE: "sphere"
SQUARE: "square"
AXIS: "axis"
NUM: "num"
VEC3: "vec3"
COLOR: "color"
MATERIAL: "material"

// identificadores, números e texto
ID: /[a-zA-Z_][a-zA-Z0-9_]*/
NUMBER: /[0-9]+(\.[0-9]+)?/
STRING: /"[^"\n]*"/

// ignorados: espaços e comentários
COMMENT: /\/\/[^\n]*/ | /\/\*(.|\n)*?\*\//
%ignore /[ \t\r\n]+/
%ignore COMMENT
"""

parser = Lark(grammar, parser="lalr", lexer="basic", strict=True)
print("Parser e lexer construídos com sucesso (gramática LALR(1) sem conflitos).")

Parser e lexer construídos com sucesso (gramática LALR(1) sem conflitos).


Para ver os tokens de um pedaço de código usamos `parser.lex()`:

In [3]:
def testar_lexer(codigo: str):
    """Exibe os tokens produzidos pelo analisador léxico da Caravela."""
    for token in parser.lex(codigo):
        print(f"{token.type:<18} {token.value!r}")

#### 2.3.1 Decisões de desenho da gramática

* **O analisador léxico é criado com `lexer="basic"`**
  * *Motivo:* Sem esta opção, o Lark não separa as palavras: `pass p { drawcam; }` seria aceite como `draw cam;`. Com `lexer="basic"`, o texto é primeiro dividido em tokens completos, e só depois analisado pela gramática.

* **A gramática é LALR(1), isto é, não é ambígua**
  * *Motivo:* O *parser* é construído com `strict=True`, que faz o Lark recusar a gramática se houver qualquer conflito LALR. Sem esta opção, o Lark resolveria os conflitos *shift/reduce* sozinho, escolhendo sempre *shift*, sem avisar.

* **Os blocos `if` e `for` usam sempre chavetas**
  * *Motivo:* Não há o problema do *else* pendente: cada `else` pertence ao `if` cujo bloco acabou de fechar. `else if` é escrito como `else` seguido de outro `if`.

* **Há um `if` por contexto (`scene_if`, `pass_if`), e não há `if` dentro de materiais**
  * *Motivo:* A sintaxe impede que um item apareça no sítio errado (ex: `draw` dentro de uma cena é um erro de sintaxe). As duas regras têm a mesma forma, mas juntá-las numa só deixaria passar esses erros para a semântica. Um `if` dentro de um material não traz nada que o `pass_if` não faça, e complicaria desnecessariamente as regras do material (dependência de unidades de textura, número de `lighting`, etc.).

* **Os ramos do `if` são um nó próprio (`scene_block`, `pass_block`)**
  * *Motivo:* Sem ele, `if c { a; b; } else { d; }` e `if c { a; } else { b; d; }` davam a mesma árvore porque o Lark descarta as chavetas (a semântica não saberia onde acaba o ramo do `if`). Com o bloco, cada ramo é um filho separado na árvore.

* **As partes opcionais usam `?`, e as alternativas com `\|` só separam formas diferentes**
  * *Motivo:* Evita repetir a mesma produção com e sem um pedaço (ex: `template_call: ID "(" args? ")" object_body` em vez de duas alternativas quase iguais). O mesmo acontece em `template_decl`, `object`, `noise` e `pass_lighting`. Em `mat_lighting`, os parâmetros substituem o `;`, usando-se `(lighting_params \| ";")`.

* **Cada propriedade diz que tipo de valor aceita (ex: `position vec3_val;`)**
  * *Motivo:* Evita erros óbvios na sintaxe, como `position 5;` ou `diffuse 3;`. Só as variáveis ficam por verificar na análise semântica (saber se `c` em `diffuse c;` é mesmo uma cor).

* **Os nomes são todos `ID`**
  * *Motivo:* A gramática não sabe se `cam` é uma câmara ou uma cena, pelo que `draw cam;` é sintaticamente válido. Esta validação de significado é tratada pela análise semântica.

* **Os nomes de cores (`white`, `red`, ...) são `ID`, não palavras reservadas**
  * *Motivo:* Reduz o número de palavras reservadas e regras. A propriedade `color` fica só como `"rgb" vec3`, e `background white;` entra por `color_val: ID`, tal como uma variável. Os nomes de cores passam a ser constantes pré-definidas no ambiente inicial, verificadas pela semântica como qualquer outro nome.

* **Os números não têm sinal**
  * *Motivo:* `-2` é o operador `SUB` aplicado a `2`, e `0..n` não é confundido com um número decimal.

### 2.4 Validação: exemplos de tokenização

#### 2.4.1 Um material
Aparecem as palavras reservadas (`MATERIAL`, `DIFFUSE`, `RGB`, ...), o nome do material, os números e a pontuação.

In [4]:
codigo_material = '''
material azul {
    diffuse rgb(0.0, 0.34, 0.72);
    lighting phong {
        shininess 60;
    }
}
'''
testar_lexer(codigo_material)

MATERIAL           'material'
ID                 'azul'
LBRACE             '{'
DIFFUSE            'diffuse'
RGB                'rgb'
LPAR               '('
NUMBER             '0.0'
COMMA              ','
NUMBER             '0.34'
COMMA              ','
NUMBER             '0.72'
RPAR               ')'
SEMICOLON          ';'
LIGHTING           'lighting'
PHONG              'phong'
LBRACE             '{'
SHININESS          'shininess'
NUMBER             '60'
SEMICOLON          ';'
RBRACE             '}'
RBRACE             '}'


#### 2.4.2 Comentários e espaços
Os comentários (`//` e `/* ... */`) e os espaços não aparecem na lista de tokens que vai para o *parser*.

In [5]:
codigo_comentarios = '''
// Material principal da cena
material vermelho {
    /* Comentário de bloco */
    diffuse rgb(1.0, 0.0, 0.0);
    lighting per_vertex;
}
'''
testar_lexer(codigo_comentarios)

MATERIAL           'material'
ID                 'vermelho'
LBRACE             '{'
DIFFUSE            'diffuse'
RGB                'rgb'
LPAR               '('
NUMBER             '1.0'
COMMA              ','
NUMBER             '0.0'
COMMA              ','
NUMBER             '0.0'
RPAR               ')'
SEMICOLON          ';'
LIGHTING           'lighting'
PER_VERTEX         'per_vertex'
SEMICOLON          ';'
RBRACE             '}'


#### 2.4.3 Palavras reservadas, identificadores e cores
`scene`, `grid`, `material` e `diffuse` são palavras reservadas, enquanto `mundo`, `azul` e a cor `blue` são `ID`.

In [6]:
codigo_identificadores = '''
scene mundo = grid length 20 divisions 2;
material azul {
    diffuse blue;
    lighting phong;
}
'''
testar_lexer(codigo_identificadores)

SCENE              'scene'
ID                 'mundo'
EQUAL              '='
GRID               'grid'
LENGTH             'length'
NUMBER             '20'
DIVISIONS          'divisions'
NUMBER             '2'
SEMICOLON          ';'
MATERIAL           'material'
ID                 'azul'
LBRACE             '{'
DIFFUSE            'diffuse'
ID                 'blue'
SEMICOLON          ';'
LIGHTING           'lighting'
PHONG              'phong'
SEMICOLON          ';'
RBRACE             '}'


#### 2.4.4 Operadores, expressões e o intervalo `0..n`
Aparecem os operadores (`CMP_OP`, `MUL_OP`), o intervalo `..` (`_RANGE`) e o número negativo em `(-1, 0.5, 2)`, que dá `SUB` seguido de `NUMBER`.

In [7]:
codigo_expressao = '''
let n = 8;
scene templo {
    for i in 0..n {
        if i % 2 == 0 {
            box p {
                translate (-1, 0.5, 2);
            }
        }
    }
}
'''
testar_lexer(codigo_expressao)

LET                'let'
ID                 'n'
EQUAL              '='
NUMBER             '8'
SEMICOLON          ';'
SCENE              'scene'
ID                 'templo'
LBRACE             '{'
FOR                'for'
ID                 'i'
IN                 'in'
NUMBER             '0'
_RANGE             '..'
ID                 'n'
LBRACE             '{'
IF                 'if'
ID                 'i'
MUL_OP             '%'
NUMBER             '2'
CMP_OP             '=='
NUMBER             '0'
LBRACE             '{'
BOX                'box'
ID                 'p'
LBRACE             '{'
TRANSLATE          'translate'
LPAR               '('
SUB                '-'
NUMBER             '1'
COMMA              ','
NUMBER             '0.5'
COMMA              ','
NUMBER             '2'
RPAR               ')'
SEMICOLON          ';'
RBRACE             '}'
RBRACE             '}'
RBRACE             '}'
RBRACE             '}'


### 2.5 Validação sintática

Testes no formato das aulas (*doctests*). `run_parser` devolve `True` se o programa for aceite, ou a posição do primeiro erro de sintaxe.

O último teste é sintaticamente correto mas tem um erro **semântico**: `cam` é uma câmara e `draw` espera uma cena. Por isso é aceite aqui, e a deteção desse erro é o assunto da secção 3.

In [8]:
import doctest
from lark.exceptions import UnexpectedInput

def run_parser(data):
  """
  Lark parser with doctests.

  >>> run_parser('camera c { position (1, 2, 3); near 0.1; }')
  True
  >>> run_parser('material m { if modo == 2 { lighting phong { shininess 100; } } else { lighting none; } }')
  'Erro de sintaxe: linha 1, coluna 14'
  >>> run_parser('camera c { position (1, 2, 3) }')
  'Erro de sintaxe: linha 1, coluna 31'
  >>> run_parser('material m { draw x; }')
  'Erro de sintaxe: linha 1, coluna 14'
  >>> run_parser('camera c { position (1, 2); }')
  'Erro de sintaxe: linha 1, coluna 26'
  >>> run_parser('scene texture = model "a.obj";')
  'Erro de sintaxe: linha 1, coluna 7'
  >>> run_parser('camera c { position 5; }')
  'Erro de sintaxe: linha 1, coluna 21'
  >>> run_parser('material m { diffuse 3; }')
  'Erro de sintaxe: linha 1, coluna 22'
  >>> run_parser('pass p { drawcam; }')
  'Erro de sintaxe: linha 1, coluna 10'
  >>> run_parser('camera c { near 0.1; far 100; fov 60; }  light l directional { direction (1, -1, 0); color grey; }')
  True
  >>> run_parser('template t(pos: vec3, c: color) { sphere s { translate pos; } }  scene sc { t((1, 0, 2), red); square chao; axis eixos { scale (5, 5, 5); } }')
  True
  >>> run_parser('material m { diffuse c; let k = 1; }')
  'Erro de sintaxe: linha 1, coluna 25'
  >>> run_parser('camera cam { position (1, 2, 3); }  pass p { draw cam; camera cam; }')
  True
  """
  try:
    parser.parse(data)
    return True
  except UnexpectedInput as e:
    return f'Erro de sintaxe: linha {e.line}, coluna {e.column}'

doctest.run_docstring_examples(run_parser, globals())

Por fim, os quatro programas da secção 1.6. São lidos diretamente deste notebook, por isso os programas testados são exatamente os que aparecem no relatório. Se o ficheiro mudar de nome, é preciso mudar também o `caminho` em `exemplos_caravela`.

In [9]:
import json

def exemplos_caravela(caminho="relatorio.ipynb", titulo="### 1.6"):
  """Devolve os blocos ``` sem linguagem indicada da secção que começa por `titulo`."""
  celulas = ["".join(c["source"]) for c in json.load(open(caminho, encoding="utf-8"))["cells"]]
  inicio = next(i for i, t in enumerate(celulas) if t.startswith(titulo))
  fim = next(i for i in range(inicio + 1, len(celulas)) if celulas[i].startswith("#"))
  blocos, atual = [], None
  for linha in "\n".join(celulas[inicio:fim]).split("\n"):
    if linha.startswith("```"):
      if atual is None:
        atual = [] if linha.strip() == "```" else False
      else:
        if atual is not False:
          blocos.append("\n".join(atual))
        atual = None
    elif atual not in (None, False):
      atual.append(linha)
  return blocos

for i, programa in enumerate(exemplos_caravela(), start=1):
  print(f"Exemplo {i}:", run_parser(programa))

Exemplo 1: True
Exemplo 2: True
Exemplo 3: True
Exemplo 4: True


### 2.6 Resumo da secção 2

- O léxico separa palavras reservadas, nomes, números e texto, e deita fora os espaços e os comentários.
- A gramática é LALR(1) e não tem conflitos (`strict=True`).
- Os erros de forma (falta de `;`, um vetor com 2 componentes, `draw` dentro de um material, ...) já são apanhados pelo *parser*, com a linha e a coluna.
- O que a gramática não consegue ver (se um nome existe, se é do tipo certo, ...) fica para a secção 3.

## 3. Especificação semântica

Na secção 1.5 listámos as propriedades da Caravela que a gramática independente de contexto não consegue verificar. Aqui especificamos a maior parte delas com uma **gramática de atributos**, escrita e executada em **LarkAG**, como na aula EX04.

### 3.1 Propriedades especificadas

| Propriedade (secção 1.5) | Na gramática de atributos | Mensagem de erro |
|---|---|---|
| Nomes declarados e visíveis | sim | `Nome não declarado` |
| Sem declarações repetidas no mesmo âmbito | sim | `Nome declarado mais de uma vez` |
| Cores pré-definidas não podem ser declaradas | sim | `As cores pré-definidas não podem ser declaradas de novo` |
| `let` locais só depois da declaração; âmbitos de `for`, `if` e `template` | sim | `Nome não declarado` |
| `let` globais sem dependências circulares | sim | `Dependência circular entre let` |
| Tipos das referências (`draw` só cenas, `layer` só texturas, ...) e das variáveis (`translate v` exige um `vec3`) | sim | `Tipo errado` |
| Argumentos de um template com o tipo dos parâmetros | sim | `Argumentos do template com tipo errado` |
| Templates não recursivos | sim | `Template recursivo` |
| Coerência do material (`shininess`/`levels`, camadas, um só `lighting`) | sim | ver 3.4 |
| Unidades de textura | sim, **calculadas** (atributo `unidades`) | — |
| Valores válidos (`near > 0`, `levels >= 2`, ...) | não | — |

Deixámos de fora os valores válidos porque, para os verificar, era preciso **avaliar** as expressões (por exemplo `near 0.1 * k`). Isso obrigava a ter um atributo `val` e um ambiente com os valores das variáveis, como no exemplo das expressões com `let` das aulas. É uma extensão direta, mas não traz nenhuma ideia nova ao que já está feito, por isso preferimos gastar o tempo no resto (fica nas melhorias, secção 4.5).

### 3.2 Ideia geral: os usos sobem, os nomes locais descem

Cada sítio onde aparece um nome produz um **uso**: o nome e o tipo que esse sítio espera. Por exemplo, `draw templo;` produz o uso `(templo, scene)`, e o `ID` dentro de uma expressão numérica (`i * 45`) produz `(i, num)`.

A Caravela tem dois tipos de nomes, e cada um é resolvido de forma diferente:

- **Nomes locais** (`let` dentro de uma cena, variável do `for`, parâmetros de um `template`) só são visíveis depois da declaração e dentro do bloco. São passados **de cima para baixo e da esquerda para a direita** num atributo herdado `env`, tal como o `env` do exemplo das aulas. Em cada item de uma cena, os usos cujo nome está em `env` são verificados logo ali. Os outros sobem.
- **Nomes globais** podem ser usados **antes** de serem declarados (secção 1.5). Por isso não podem ser passados para baixo num atributo herdado: o ambiente global só fica completo depois de visitar o programa todo, e o LarkAG não visita o mesmo nó duas vezes. Os usos que não foram resolvidos localmente sobem até à raiz, num atributo sintetizado `usos`, e é na produção `start` que são comparados com a tabela de símbolos global (`tabela`), que também é sintetizada.

Assim, cada propriedade é verificada no primeiro nó onde há informação suficiente para isso: os nomes locais no item da cena, os globais na raiz e a coerência do material no nó `materialdecl`.

### 3.3 Atributos

| Atributo | Tipo | Em que símbolos | O que representa |
|---|---|---|---|
| `usos` | sintetizado | quase todos | lista de usos `(nome, tipo esperado, tipos dos argumentos)` que ainda falta verificar |
| `nomes` | sintetizado | declarações de topo, `start` | lista `(nome, tipo, dependências)` dos nomes globais declarados |
| `tabela` | sintetizado | `start` | tabela de símbolos global: nome ↦ tipo, já com as cores pré-definidas |
| `env` | **herdado** | `sceneitems`, `sceneitem`, `scenefor`, `sceneif`, `sceneblock` | nomes locais visíveis nesse ponto: nome ↦ tipo |
| `locais` | sintetizado | `sceneitems`, `sceneitem`, `params` | nomes locais declarados até ali |
| `nome`, `tipo` | sintetizado | `num`, `term`, `factor`, `value`, `letdecl`, `param` | nome de uma expressão que é só um `ID`, e o tipo de um valor |
| `modo` | sintetizado / **herdado** | `lightingmode`, `matlighting` / `lightingparams` | modo de iluminação; desce até aos parâmetros para os validar |
| `modos`, `papeis` | sintetizado | `materialdecl` | modos de iluminação e papéis das camadas do material |
| `unidades` | sintetizado | `materialdecl`, `decl`, `start` (`materiais`) | unidade de textura de cada camada: papel ↦ 0, 1, 2, ... |

Os tipos são os da secção 1.4: `num`, `vec3`, `color`, `material`, `scene`, `camera`, `light`, `texture`, `pass` e, para os templates, `(template, tipos dos parâmetros)`. Um `let x = y;` tem tipo `ref(y)`: o seu tipo é o de `y`, que pode ainda não estar declarado.

**Um `ID` sozinho num valor.** Na gramática, `let c = red;` e `pilar(3, azul)` lêem `red` e `azul` como números (`value → num → term → factor → ID`). O atributo `nome` resolve isto: sobe de `factor` até `value` enquanto a expressão for só um `ID`. Em `value`, um `num` com `nome` é tratado como uma referência ao nome, de qualquer tipo de valor (`any`), e não como um número.

### 3.4 Regras principais

Notação das aulas: os índices distinguem as ocorrências do mesmo símbolo, e `⊕` é a extensão de um ambiente. `resto(usos, env)` são os usos cujo nome não está em `env` (sobem), e `ok(usos, env)` diz que os usos cujo nome está em `env` têm o tipo esperado.

| Produção | Regras de avaliação | Condições de contexto |
|---|---|---|
| `start → project? decl*` | `start.nomes = ⋃ decl.nomes` | os nomes de `start.nomes` não se repetem e nenhum é uma cor |
| | `start.usos = ⋃ decl.usos` | o grafo de dependências dos `let` não tem ciclos |
| | `start.tabela = PRE ⊕ start.nomes` (com as `ref` resolvidas) | o grafo de chamadas dos templates não tem ciclos |
| | `start.materiais = ⋃ decl.unidades` | ∀ `(n, t) ∈ start.usos`: `n ∈ start.tabela` e `start.tabela(n)` é compatível com `t` |
| `sceneitems₀ → sceneitems₁ sceneitem` | `sceneitems₁.env = sceneitems₀.env` | `dom(sceneitem.locais) ∩ dom(sceneitems₁.locais) = ∅` |
| | `sceneitem.env = sceneitems₀.env ⊕ sceneitems₁.locais` | nenhum nome de `sceneitem.locais` é uma cor |
| | `sceneitems₀.locais = sceneitems₁.locais ⊕ sceneitem.locais` | |
| | `sceneitems₀.usos = sceneitems₁.usos ++ sceneitem.usos` | |
| `sceneitems → ε` | `sceneitems.locais = {}`, `sceneitems.usos = []` | |
| `sceneitem → letdecl` | `sceneitem.locais = {letdecl.nome ↦ letdecl.tipo}` | `ok(letdecl.usos, sceneitem.env)` |
| | `sceneitem.usos = resto(letdecl.usos, sceneitem.env)` | |
| `sceneitem → object` (igual para `templatecall`, `scenefor`, `sceneif`) | `sceneitem.locais = {}`, `sceneitem.usos = resto(object.usos, sceneitem.env)` | `ok(object.usos, sceneitem.env)` |
| `scenefor → for ID in num₁ .. num₂ { sceneitems }` | `sceneitems.env = scenefor.env ⊕ {ID ↦ num}` | |
| | `scenefor.usos = num₁.usos ++ num₂.usos ++ sceneitems.usos` | |
| `sceneblock → { sceneitems }` | `sceneitems.env = sceneblock.env` | |
| `templatedecl → template ID ( params ) { sceneitems }` | `sceneitems.env = params.locais` | |
| | `templatedecl.nomes = [(ID, (template, params.tipos), templates chamados)]` | |
| `factor → ID` | `factor.usos = [(ID, num)]`, `factor.nome = ID` | |
| `value → num` | `value.tipo = ref(num.nome)` se `num.nome`, senão `num` | |
| | `value.usos = [(num.nome, any)]` se `num.nome`, senão `num.usos` | |
| `draw → draw idlist ;` | `draw.usos = [(n, scene) \| n ∈ idlist.nomes]` | |
| `layersource → ID` | `layersource.usos = [(ID, texture)]`, `layersource.textura = True` | |
| `materialdecl → material ID { matitem* }` | `materialdecl.modos = [i.modo \| i ∈ matitem, i.modo ≠ ⊥]` | `\|modos\| ≤ 1` |
| | `materialdecl.papeis = [i.papel \| i ∈ matitem, i.papel ≠ ⊥]` | os papéis não se repetem |
| | `materialdecl.unidades = {p ↦ k}`, com `k` a posição de `p` entre as camadas com textura | `specular ∈ papeis ⇒ modos = [phong]` |
| | | `night ∈ papeis ⇒ none ∉ modos` |
| | | `papeis ≠ [] ⇒ day ∈ papeis` |
| `matlighting → lighting lightingmode lightingparams` | `lightingparams.modo = lightingmode.modo` | |
| `lightingparams → { lightingparam* }` | `lightingparams.nomes = [p.nome \| p ∈ lightingparam]` | `shininess ∈ nomes ⇒ lightingparams.modo = phong` |
| | | `levels ∈ nomes ⇒ lightingparams.modo = toon` |

As restantes produções só copiam `usos` dos filhos para o pai (por exemplo, `vec3.usos = num₁.usos ++ num₂.usos ++ num₃.usos`). Estão todas na implementação em 3.6.

**Dependências.** O `env` de cada item depende dos `locais` sintetizados pelos itens à sua esquerda, por isso `sceneitems` é uma gramática **L-atribuída**: os atributos podem ser calculados numa só visita, da esquerda para a direita. Por exemplo, em

```
scene s { let a = 1; box b { translate (a, 0, 0); } }
```

o primeiro item sintetiza `locais = {a ↦ num}`, que passa a fazer parte do `env` herdado do segundo item. O uso `(a, num)` que sobe do `translate` é resolvido nesse item e não chega à raiz. Se os dois itens estivessem pela ordem inversa, `a` não estaria em `env`, o uso subiria até `start`, e daria `Nome não declarado` (a não ser que houvesse um `a` global). Nas declarações de topo é diferente: a ordem não importa, porque todos os nomes globais são juntados em `start.tabela` antes de os usos serem verificados.

### 3.5 Da gramática da secção 2 para o LarkAG

A gramática de atributos usa a gramática da secção 2 com quatro transformações, que não mudam a linguagem aceite (verificado em 3.7):

1. **Nomes sem `_`.** O LarkAG dá às alternativas de cada regra os nomes `regra_0`, `regra_1`, ... e depois identifica a regra pelo que está antes do primeiro `_`. Por isso `let_decl` passa a `letdecl`, `scene_item` a `sceneitem`, e assim por diante. Os terminais (`PER_VERTEX`, `CMP_OP`) não têm este problema e ficam iguais.
2. **Listas que recebem um atributo herdado são recursivas à esquerda.** Num `item*`, o LarkAG só dá acesso à lista inteira (`@item`), e não permite dar um atributo herdado a cada elemento. Os itens das cenas precisam do `env`, por isso `scene_item*` passa a `sceneitems: sceneitems sceneitem | ε`, como `decls` no exemplo das aulas. As outras listas (`decl*`, `matitem*`, `pass_item*`, ...) só sintetizam atributos e ficam com `*`, lidas com `@`.
3. **O intervalo `..` volta a ser um literal.** O LarkAG não aceita terminais cujo nome começa por `_`, como o `_RANGE` da secção 2. Em `scenefor` escreve-se o literal `".."`, que também não aparece na árvore.
4. **Partes opcionais com atributos passam a alternativas.** Uma regra só pode referir um filho que existe sempre. Por isso `object: shape ID (":" ID)? object_body` passa a duas alternativas, uma com e outra sem material. O mesmo acontece em `template_call`, `template_decl`, `noise`, `mat_lighting`, `pass_lighting` e nos `if` (`if`, `if ... else`, `if ... else if`).

**Como é executada.** O `Lark_AG(...).process` das aulas usa o *parser* por omissão do Lark (Earley). Para usar exatamente o mesmo *parser* da secção 2 (LALR com `lexer="basic"`), a função `analisar` constrói-o a partir da gramática gerada pelo LarkAG (`getCFG()`) e executa o *Interpreter* que o LarkAG gera (`getInterpreterFile()`). Este é executado num espaço de nomes que contém as funções auxiliares de 3.6, para que as regras as possam chamar.

### 3.6 Implementação em LarkAG

As regras usam algumas funções auxiliares em Python: os nomes dos tipos, o ambiente inicial com as cores, a resolução dos usos num ambiente local, a tabela global e a deteção de ciclos.

In [10]:
!pip -q install --no-cache-dir "git+https://github.com/nmacedo/LarkAG.git@ag-soundness-fixes"

In [11]:
# Tipos e funções auxiliares usados nas regras da gramática de atributos.
# Um "uso" é um tuplo (nome, tipo esperado, tipos dos argumentos ou None).

K_NUM, K_VEC3, K_COLOR, K_MATERIAL = "num", "vec3", "color", "material"
K_SCENE, K_CAMERA, K_LIGHT, K_TEXTURE = "scene", "camera", "light", "texture"
K_TEMPLATE, K_PASS, K_ANY, REF = "template", "pass", "any", "ref"
K_NONE, K_PHONG, K_TOON = "none", "phong", "toon"
K_DAY, K_SPECULAR, K_NIGHT = "day", "specular", "night"
K_SHININESS, K_LEVELS = "shininess", "levels"

# ambiente inicial: as cores pré-definidas
PRE = {c: K_COLOR for c in ["white", "black", "red", "green", "blue", "yellow", "grey"]}

# tipos que um valor (let, argumento de template) pode ter
VALORES = {K_NUM, K_VEC3, K_COLOR, K_MATERIAL}

def ref(t):
  """O tipo é uma referência a outro nome, ainda por resolver (let x = y;)."""
  return isinstance(t, tuple) and t[0] == REF

def compativel(t, esperado):
  if esperado == K_ANY:
    return t in VALORES
  if esperado == K_TEMPLATE:
    return isinstance(t, tuple) and t[0] == K_TEMPLATE
  return t == esperado

def _resolver(usos, env):
  """Resolve os usos com os nomes locais de env. Devolve (usos que sobem, nomes com tipo errado)."""
  resto, erros = [], []
  for nome, esperado, args in usos:
    if args is not None:
      args = tuple(env.get(t[1], t) if ref(t) else t for t in args)
    if nome not in env:
      resto.append((nome, esperado, args))
    elif ref(env[nome]):
      resto.append((env[nome][1], esperado, args))
    elif not compativel(env[nome], esperado):
      erros.append(nome)
  return resto, erros

def resto(usos, env):
  return _resolver(usos, env)[0]

def tipos_ok(usos, env):
  return not _resolver(usos, env)[1]

def tipo_local(t, env):
  return env.get(t[1], t) if ref(t) else t

def tipos_globais(nomes):
  """Tabela de símbolos global: nome -> tipo, com as referências entre let resolvidas."""
  brutos = dict(PRE)
  brutos.update({n: t for n, t, deps in nomes})
  def tipo(n, vistos):
    t = brutos.get(n)
    if ref(t):
      return None if t[1] in vistos else tipo(t[1], vistos | {t[1]})
    return t
  return {n: tipo(n, {n}) for n in brutos}

def sem_repetidos(lista):
  return len(lista) == len(set(lista))

def tem_ciclo(grafo):
  estado = {}
  def visita(n):
    if estado.get(n) == 1:
      return True
    if estado.get(n) == 2:
      return False
    estado[n] = 1
    if any(visita(m) for m in grafo[n] if m in grafo):
      return True
    estado[n] = 2
    return False
  return any(visita(n) for n in grafo)

def argumentos_ok(usos, tabela):
  for nome, esperado, args in usos:
    if esperado == K_TEMPLATE and compativel(tabela.get(nome), K_TEMPLATE):
      reais = tuple(tabela.get(t[1]) if ref(t) else t for t in args)
      if None not in reais and reais != tabela[nome][1]:
        return False
  return True

In [12]:
from lark_ag import Lark_AG, ContextConditionError

ag_grammar = r"""
// ---------- Atributos ----------
SA(start)          { nomes : list, usos : list, tabela : dict, materiais : dict }
SA(decl)           { nomes : list, usos : list, unidades : dict }

SA(letdecl)        { nome : str, tipo : any, usos : list }
SA(texturedecl)    { nomes : list }
SA(materialdecl)   { nomes : list, usos : list, modos : list, papeis : list, unidades : dict }
SA(matitem)        { usos : list, modo : str, papel : str, textura : bool }
SA(diffuse)        { usos : list }
SA(matlighting)    { usos : list, modo : str }
SA(lightingmode)   { modo : str }
SA(lightingparams) { usos : list, nomes : list }
IA(lightingparams) { modo : str }
SA(lightingparam)  { usos : list, nome : str }
SA(shininess)      { usos : list }
SA(levels)         { usos : list }
SA(layer)          { usos : list, papel : str, textura : bool }
SA(layerrole)      { papel : str }
SA(layersource)    { usos : list, textura : bool }
SA(noise)          { usos : list }

SA(scenedecl)      { nomes : list, usos : list }
SA(scenesource)    { usos : list }
SA(grid)           { usos : list }
SA(sceneitems)     { usos : list, locais : dict }
IA(sceneitems)     { env : dict }
SA(sceneitem)      { usos : list, locais : dict }
IA(sceneitem)      { env : dict }
SA(object)         { usos : list }
SA(templatecall)   { usos : list }
SA(objectbody)     { usos : list }
SA(transform)      { usos : list }
SA(translate)      { usos : list }
SA(scale)          { usos : list }
SA(rotate)         { usos : list }
SA(scenefor)       { usos : list }
IA(scenefor)       { env : dict }
SA(sceneif)        { usos : list }
IA(sceneif)        { env : dict }
SA(sceneblock)     { usos : list }
IA(sceneblock)     { env : dict }
SA(templatedecl)   { nomes : list, usos : list }
SA(params)         { locais : dict, tipos : tuple }
SA(param)          { nome : str, tipo : str }
SA(type)           { tipo : str }
SA(args)           { usos : list, tipos : tuple }

SA(cameradecl)     { nomes : list, usos : list }
SA(cameraprop)     { usos : list }
SA(position)       { usos : list }
SA(lookat)         { usos : list }
SA(direction)      { usos : list }
SA(near)           { usos : list }
SA(far)            { usos : list }
SA(fov)            { usos : list }
SA(lightdecl)      { nomes : list, usos : list }
SA(lightprop)      { usos : list }
SA(lightcolor)     { usos : list }
SA(backgrounddecl) { usos : list }

SA(passdecl)       { nomes : list, usos : list }
SA(passitem)       { usos : list }
SA(draw)           { usos : list }
SA(passcamera)     { usos : list }
SA(passlights)     { usos : list }
SA(passlighting)   { usos : list }
SA(passmaterial)   { usos : list }
SA(passif)         { usos : list }
SA(passblock)      { usos : list }
SA(idlist)         { nomes : list }

SA(num)            { usos : list, nome : str }
SA(term)           { usos : list, nome : str }
SA(factor)         { usos : list, nome : str }
SA(vec3)           { usos : list }
SA(color)          { usos : list }
SA(cond)           { usos : list }
SA(vec3val)        { usos : list }
SA(colorval)       { usos : list }
SA(value)          { usos : list, tipo : any }

// ---------- Programa ----------
start : project? decl*
ER {
  start.nomes = sum([d.nomes for d in @decl], []);
  start.usos = sum([d.usos for d in @decl], []);
  start.tabela = tipos_globais(start.nomes);
  start.materiais = {m: u for d in @decl for m, u in d.unidades.items()};
}
CC {
  sem_repetidos([n for n, t, deps in start.nomes]) -> "Nome declarado mais de uma vez";
  not (set(n for n, t, deps in start.nomes) & set(PRE)) -> "As cores pré-definidas não podem ser declaradas de novo";
  not tem_ciclo({n: deps for n, t, deps in start.nomes if not compativel(t, K_TEMPLATE)}) -> "Dependência circular entre let";
  not tem_ciclo({n: deps for n, t, deps in start.nomes if compativel(t, K_TEMPLATE)}) -> "Template recursivo";
  all(n in start.tabela for n, e, a in start.usos) -> "Nome não declarado";
  all(compativel(start.tabela[n], e) for n, e, a in start.usos if start.tabela.get(n) is not None) -> "Tipo errado";
  argumentos_ok(start.usos, start.tabela) -> "Argumentos do template com tipo errado";
}

project : "project" STRING ";"

decl : letdecl
ER { decl.nomes = [(letdecl.nome, letdecl.tipo, set(n for n, e, a in letdecl.usos))]; decl.usos = letdecl.usos; decl.unidades = {}; }
     | texturedecl
ER { decl.nomes = texturedecl.nomes; decl.usos = []; decl.unidades = {}; }
     | materialdecl
ER { decl.nomes = materialdecl.nomes; decl.usos = materialdecl.usos; decl.unidades = materialdecl.unidades; }
     | scenedecl
ER { decl.nomes = scenedecl.nomes; decl.usos = scenedecl.usos; decl.unidades = {}; }
     | templatedecl
ER { decl.nomes = templatedecl.nomes; decl.usos = templatedecl.usos; decl.unidades = {}; }
     | cameradecl
ER { decl.nomes = cameradecl.nomes; decl.usos = cameradecl.usos; decl.unidades = {}; }
     | lightdecl
ER { decl.nomes = lightdecl.nomes; decl.usos = lightdecl.usos; decl.unidades = {}; }
     | backgrounddecl
ER { decl.nomes = []; decl.usos = backgrounddecl.usos; decl.unidades = {}; }
     | passdecl
ER { decl.nomes = passdecl.nomes; decl.usos = passdecl.usos; decl.unidades = {}; }

letdecl : "let" ID "=" value ";"
ER { letdecl.nome = str(ID); letdecl.tipo = value.tipo; letdecl.usos = value.usos; }

// ---------- Texturas e materiais ----------
texturedecl : "texture" ID "=" STRING MIPMAP? ";"
ER { texturedecl.nomes = [(str(ID), K_TEXTURE, set())]; }

materialdecl : "material" ID "{" matitem* "}"
ER {
  materialdecl.modos = [i.modo for i in @matitem if i.modo is not None];
  materialdecl.papeis = [i.papel for i in @matitem if i.papel is not None];
  materialdecl.unidades = {str(ID): {p: u for u, p in enumerate([i.papel for i in @matitem if i.textura])}};
  materialdecl.nomes = [(str(ID), K_MATERIAL, set())];
  materialdecl.usos = sum([i.usos for i in @matitem], []);
}
CC {
  len(materialdecl.modos) <= 1 -> "Um material só pode ter um lighting";
  sem_repetidos(materialdecl.papeis) -> "Camada repetida no material";
  K_SPECULAR not in materialdecl.papeis or materialdecl.modos == [K_PHONG] -> "A camada specular exige lighting phong";
  K_NIGHT not in materialdecl.papeis or K_NONE not in materialdecl.modos -> "A camada night exige um modo com luz";
  K_DAY in materialdecl.papeis or not materialdecl.papeis -> "As camadas specular, night e clouds exigem uma camada day";
}

matitem : diffuse
ER { matitem.usos = diffuse.usos; matitem.modo = None; matitem.papel = None; matitem.textura = False; }
        | matlighting
ER { matitem.usos = matlighting.usos; matitem.modo = matlighting.modo; matitem.papel = None; matitem.textura = False; }
        | layer
ER { matitem.usos = layer.usos; matitem.modo = None; matitem.papel = layer.papel; matitem.textura = layer.textura; }

diffuse : "diffuse" colorval ";"
ER { diffuse.usos = colorval.usos; }

matlighting : "lighting" lightingmode lightingparams
ER { lightingparams.modo = lightingmode.modo; matlighting.modo = lightingmode.modo; matlighting.usos = lightingparams.usos; }
            | "lighting" lightingmode ";"
ER { matlighting.modo = lightingmode.modo; matlighting.usos = []; }

lightingmode : NONE
ER { lightingmode.modo = str(NONE); }
             | PER_VERTEX
ER { lightingmode.modo = str(PER_VERTEX); }
             | PHONG
ER { lightingmode.modo = str(PHONG); }
             | TOON
ER { lightingmode.modo = str(TOON); }

lightingparams : "{" lightingparam* "}"
ER { lightingparams.nomes = [p.nome for p in @lightingparam]; lightingparams.usos = sum([p.usos for p in @lightingparam], []); }
CC {
  K_SHININESS not in lightingparams.nomes or lightingparams.modo == K_PHONG -> "shininess só existe em lighting phong";
  K_LEVELS not in lightingparams.nomes or lightingparams.modo == K_TOON -> "levels só existe em lighting toon";
}

lightingparam : shininess
ER { lightingparam.nome = K_SHININESS; lightingparam.usos = shininess.usos; }
              | levels
ER { lightingparam.nome = K_LEVELS; lightingparam.usos = levels.usos; }

shininess : "shininess" num ";"
ER { shininess.usos = num.usos; }

levels : "levels" num ";"
ER { levels.usos = num.usos; }

layer : "layer" layerrole layersource ";"
ER { layer.papel = layerrole.papel; layer.textura = layersource.textura; layer.usos = layersource.usos; }

layerrole : DAY
ER { layerrole.papel = str(DAY); }
          | SPECULAR
ER { layerrole.papel = str(SPECULAR); }
          | NIGHT
ER { layerrole.papel = str(NIGHT); }
          | CLOUDS
ER { layerrole.papel = str(CLOUDS); }

layersource : ID
ER { layersource.usos = [(str(ID), K_TEXTURE, None)]; layersource.textura = True; }
            | noise
ER { layersource.usos = noise.usos; layersource.textura = False; }

noise : "noise" "speed" num
ER { noise.usos = num.usos; }
      | "noise"
ER { noise.usos = []; }

// ---------- Cenas e templates ----------
scenedecl : "scene" ID "=" scenesource ";"
ER { scenedecl.nomes = [(str(ID), K_SCENE, set())]; scenedecl.usos = scenesource.usos; }
          | "scene" ID "{" sceneitems "}"
ER { sceneitems.env = {}; scenedecl.nomes = [(str(ID), K_SCENE, set())]; scenedecl.usos = sceneitems.usos; }

scenesource : model
ER { scenesource.usos = []; }
            | grid
ER { scenesource.usos = grid.usos; }

model : "model" STRING modeloption*

modeloption : UNITIZE
            | SWAP_YZ

grid : "grid" "length" num "divisions" num
ER { grid.usos = num[1].usos + num[2].usos; }

sceneitems : sceneitems sceneitem
ER {
  sceneitems[2].env = sceneitems[1].env;
  sceneitem.env = sceneitems[1].env | sceneitems[2].locais;
  sceneitems[1].locais = sceneitems[2].locais | sceneitem.locais;
  sceneitems[1].usos = sceneitems[2].usos + sceneitem.usos;
}
CC {
  not (set(sceneitem.locais) & set(sceneitems[2].locais)) -> "Nome declarado mais de uma vez";
  not (set(sceneitem.locais) & set(PRE)) -> "As cores pré-definidas não podem ser declaradas de novo";
}
           |
ER { sceneitems.locais = {}; sceneitems.usos = []; }

sceneitem : object
ER { sceneitem.usos = resto(object.usos, sceneitem.env); sceneitem.locais = {}; }
CC { tipos_ok(object.usos, sceneitem.env) -> "Tipo errado"; }
          | templatecall
ER { sceneitem.usos = resto(templatecall.usos, sceneitem.env); sceneitem.locais = {}; }
CC { tipos_ok(templatecall.usos, sceneitem.env) -> "Tipo errado"; }
          | letdecl
ER { sceneitem.usos = resto(letdecl.usos, sceneitem.env); sceneitem.locais = {letdecl.nome: tipo_local(letdecl.tipo, sceneitem.env)}; }
CC { tipos_ok(letdecl.usos, sceneitem.env) -> "Tipo errado"; }
          | scenefor
ER { scenefor.env = sceneitem.env; sceneitem.usos = resto(scenefor.usos, sceneitem.env); sceneitem.locais = {}; }
CC { tipos_ok(scenefor.usos, sceneitem.env) -> "Tipo errado"; }
          | sceneif
ER { sceneif.env = sceneitem.env; sceneitem.usos = resto(sceneif.usos, sceneitem.env); sceneitem.locais = {}; }
CC { tipos_ok(sceneif.usos, sceneitem.env) -> "Tipo errado"; }

object : shape ID ":" ID objectbody
ER { object.usos = [(str(@ID[1]), K_MATERIAL, None)] + objectbody.usos; }
       | shape ID objectbody
ER { object.usos = objectbody.usos; }

shape : BOX
      | SPHERE
      | SQUARE
      | AXIS

templatecall : ID "(" args ")" objectbody
ER { templatecall.usos = [(str(ID), K_TEMPLATE, args.tipos)] + args.usos + objectbody.usos; }
             | ID "(" ")" objectbody
ER { templatecall.usos = [(str(ID), K_TEMPLATE, ())] + objectbody.usos; }

objectbody : "{" transform* "}"
ER { objectbody.usos = sum([t.usos for t in @transform], []); }
           | ";"
ER { objectbody.usos = []; }

transform : translate
ER { transform.usos = translate.usos; }
          | scale
ER { transform.usos = scale.usos; }
          | rotate
ER { transform.usos = rotate.usos; }

translate : "translate" vec3val ";"
ER { translate.usos = vec3val.usos; }

scale : "scale" vec3val ";"
ER { scale.usos = vec3val.usos; }

rotate : "rotate" num "around" vec3val ";"
ER { rotate.usos = num.usos + vec3val.usos; }

scenefor : "for" ID "in" num ".." num "{" sceneitems "}"
ER { sceneitems.env = scenefor.env | {str(ID): K_NUM}; scenefor.usos = num[1].usos + num[2].usos + sceneitems.usos; }
CC { str(ID) not in PRE -> "As cores pré-definidas não podem ser declaradas de novo"; }

sceneif : "if" cond sceneblock
ER { sceneblock.env = sceneif.env; sceneif.usos = cond.usos + sceneblock.usos; }
        | "if" cond sceneblock "else" sceneblock
ER { sceneblock[1].env = sceneif.env; sceneblock[2].env = sceneif.env; sceneif.usos = cond.usos + sceneblock[1].usos + sceneblock[2].usos; }
        | "if" cond sceneblock "else" sceneif
ER { sceneblock.env = sceneif[1].env; sceneif[2].env = sceneif[1].env; sceneif[1].usos = cond.usos + sceneblock.usos + sceneif[2].usos; }

sceneblock : "{" sceneitems "}"
ER { sceneitems.env = sceneblock.env; sceneblock.usos = sceneitems.usos; }

templatedecl : "template" ID "(" params ")" "{" sceneitems "}"
ER {
  sceneitems.env = params.locais;
  templatedecl.nomes = [(str(ID), (K_TEMPLATE, params.tipos), set(n for n, e, a in sceneitems.usos if e == K_TEMPLATE))];
  templatedecl.usos = sceneitems.usos;
}
             | "template" ID "(" ")" "{" sceneitems "}"
ER {
  sceneitems.env = {};
  templatedecl.nomes = [(str(ID), (K_TEMPLATE, ()), set(n for n, e, a in sceneitems.usos if e == K_TEMPLATE))];
  templatedecl.usos = sceneitems.usos;
}

params : params "," param
ER { params[1].locais = params[2].locais | {param.nome: param.tipo}; params[1].tipos = params[2].tipos + (param.tipo,); }
CC {
  param.nome not in params[2].locais -> "Nome declarado mais de uma vez";
  param.nome not in PRE -> "As cores pré-definidas não podem ser declaradas de novo";
}
       | param
ER { params.locais = {param.nome: param.tipo}; params.tipos = (param.tipo,); }
CC { param.nome not in PRE -> "As cores pré-definidas não podem ser declaradas de novo"; }

param : ID ":" type
ER { param.nome = str(ID); param.tipo = type.tipo; }

type : NUM
ER { type.tipo = str(NUM); }
     | VEC3
ER { type.tipo = str(VEC3); }
     | COLOR
ER { type.tipo = str(COLOR); }
     | MATERIAL
ER { type.tipo = str(MATERIAL); }

args : args "," value
ER { args[1].tipos = args[2].tipos + (value.tipo,); args[1].usos = args[2].usos + value.usos; }
     | value
ER { args.tipos = (value.tipo,); args.usos = value.usos; }

// ---------- Câmara, luz e fundo ----------
cameradecl : "camera" ID "{" cameraprop* "}"
ER { cameradecl.nomes = [(str(ID), K_CAMERA, set())]; cameradecl.usos = sum([p.usos for p in @cameraprop], []); }

cameraprop : position
ER { cameraprop.usos = position.usos; }
           | lookat
ER { cameraprop.usos = lookat.usos; }
           | direction
ER { cameraprop.usos = direction.usos; }
           | near
ER { cameraprop.usos = near.usos; }
           | far
ER { cameraprop.usos = far.usos; }
           | fov
ER { cameraprop.usos = fov.usos; }

position : "position" vec3val ";"
ER { position.usos = vec3val.usos; }

lookat : "look_at" vec3val ";"
ER { lookat.usos = vec3val.usos; }

direction : "direction" vec3val ";"
ER { direction.usos = vec3val.usos; }

near : "near" num ";"
ER { near.usos = num.usos; }

far : "far" num ";"
ER { far.usos = num.usos; }

fov : "fov" num ";"
ER { fov.usos = num.usos; }

lightdecl : "light" ID "directional" "{" lightprop* "}"
ER { lightdecl.nomes = [(str(ID), K_LIGHT, set())]; lightdecl.usos = sum([p.usos for p in @lightprop], []); }

lightprop : direction
ER { lightprop.usos = direction.usos; }
          | lightcolor
ER { lightprop.usos = lightcolor.usos; }

lightcolor : "color" colorval ";"
ER { lightcolor.usos = colorval.usos; }

backgrounddecl : "background" colorval ";"
ER { backgrounddecl.usos = colorval.usos; }

// ---------- Passes ----------
passdecl : "pass" ID "{" passitem* "}"
ER { passdecl.nomes = [(str(ID), K_PASS, set())]; passdecl.usos = sum([p.usos for p in @passitem], []); }

passitem : draw
ER { passitem.usos = draw.usos; }
         | passcamera
ER { passitem.usos = passcamera.usos; }
         | passlights
ER { passitem.usos = passlights.usos; }
         | passlighting
ER { passitem.usos = passlighting.usos; }
         | passmaterial
ER { passitem.usos = passmaterial.usos; }
         | passif
ER { passitem.usos = passif.usos; }

draw : "draw" idlist ";"
ER { draw.usos = [(n, K_SCENE, None) for n in idlist.nomes]; }

passcamera : "camera" ID ";"
ER { passcamera.usos = [(str(ID), K_CAMERA, None)]; }

passlights : "lights" idlist ";"
ER { passlights.usos = [(n, K_LIGHT, None) for n in idlist.nomes]; }

passlighting : "lighting" lightingmode lightingparams "for" "all" ";"
ER { lightingparams.modo = lightingmode.modo; passlighting.usos = lightingparams.usos; }
             | "lighting" lightingmode "for" "all" ";"
ER { passlighting.usos = []; }

passmaterial : "material" ID "for" "all" ";"
ER { passmaterial.usos = [(str(ID), K_MATERIAL, None)]; }

passif : "if" cond passblock
ER { passif.usos = cond.usos + passblock.usos; }
       | "if" cond passblock "else" passblock
ER { passif.usos = cond.usos + passblock[1].usos + passblock[2].usos; }
       | "if" cond passblock "else" passif
ER { passif[1].usos = cond.usos + passblock.usos + passif[2].usos; }

passblock : "{" passitem* "}"
ER { passblock.usos = sum([p.usos for p in @passitem], []); }

idlist : idlist "," ID
ER { idlist[1].nomes = idlist[2].nomes + [str(ID)]; }
       | ID
ER { idlist.nomes = [str(ID)]; }

// ---------- Valores ----------
num : num ADD term
ER { num[1].usos = num[2].usos + term.usos; num[1].nome = None; }
    | num SUB term
ER { num[1].usos = num[2].usos + term.usos; num[1].nome = None; }
    | term
ER { num.usos = term.usos; num.nome = term.nome; }

term : term MUL_OP factor
ER { term[1].usos = term[2].usos + factor.usos; term[1].nome = None; }
     | factor
ER { term.usos = factor.usos; term.nome = factor.nome; }

factor : NUMBER
ER { factor.usos = []; factor.nome = None; }
       | ID
ER { factor.usos = [(str(ID), K_NUM, None)]; factor.nome = str(ID); }
       | SUB factor
ER { factor[1].usos = factor[2].usos; factor[1].nome = None; }
       | "(" num ")"
ER { factor.usos = num.usos; factor.nome = None; }

vec3 : "(" num "," num "," num ")"
ER { vec3.usos = num[1].usos + num[2].usos + num[3].usos; }

color : "rgb" vec3
ER { color.usos = vec3.usos; }

cond : num CMP_OP num
ER { cond.usos = num[1].usos + num[2].usos; }

vec3val : vec3
ER { vec3val.usos = vec3.usos; }
        | ID
ER { vec3val.usos = [(str(ID), K_VEC3, None)]; }

colorval : color
ER { colorval.usos = color.usos; }
         | ID
ER { colorval.usos = [(str(ID), K_COLOR, None)]; }

value : num
ER { value.tipo = (REF, num.nome) if num.nome else K_NUM; value.usos = [(num.nome, K_ANY, None)] if num.nome else num.usos; }
      | vec3
ER { value.tipo = K_VEC3; value.usos = vec3.usos; }
      | color
ER { value.tipo = K_COLOR; value.usos = color.usos; }

// ---------- Léxico (igual ao da secção 2) ----------
CMP_OP: "==" | "!=" | "<=" | ">=" | "<" | ">"
ADD: "+"
SUB: "-"
MUL_OP: "*" | "/" | "%"

NONE: "none"
PER_VERTEX: "per_vertex"
PHONG: "phong"
TOON: "toon"
DAY: "day"
SPECULAR: "specular"
NIGHT: "night"
CLOUDS: "clouds"
UNITIZE: "unitize"
SWAP_YZ: "swap_yz"
MIPMAP: "mipmap"
BOX: "box"
SPHERE: "sphere"
SQUARE: "square"
AXIS: "axis"
NUM: "num"
VEC3: "vec3"
COLOR: "color"
MATERIAL: "material"

ID: /[a-zA-Z_][a-zA-Z0-9_]*/
NUMBER: /[0-9]+(\.[0-9]+)?/
STRING: /"[^"\n]*"/

COMMENT: /\/\/[^\n]*/ | /\/\*(.|\n)*?\*\//
%ignore /[ \t\r\n]+/
%ignore COMMENT
"""

ag = Lark_AG(ag_grammar)

In [13]:
ag_parser = Lark(ag.getCFG(), parser="lalr", lexer="basic")
ag_ns = dict(globals())
exec(ag.getInterpreterFile(), ag_ns)

def analisar(data):
  """Analisa um programa em Caravela e devolve a raiz da árvore, já com os atributos calculados."""
  tree = ag_parser.parse(data)
  ag_ns["MyInterpreter"]().visit(tree)
  return tree

### 3.7 Validação

Primeiro, a gramática transformada aceita exatamente os mesmos programas que a da secção 2: os testes de 2.5 e os quatro exemplos dão o mesmo resultado nos dois *parsers*.

In [14]:
import re

def aceita(p, data):
  try:
    p.parse(data)
    return True
  except UnexpectedInput:
    return False

testes = re.findall(r">>> run_parser\('(.*)'\)", run_parser.__doc__) + exemplos_caravela()
print(all(aceita(parser, t) == aceita(ag_parser, t) for t in testes), f"({len(testes)} programas)")

True (17 programas)


Os testes semânticos usam os erros referidos no exemplo 3 (secção 1.6) e um caso para cada propriedade de 3.1. `run_semantica` devolve `True` se o programa estiver correto, ou a mensagem do primeiro erro encontrado.

In [15]:
def run_semantica(data):
  """
  Análise semântica com doctests.

  Os erros do exemplo 3:

  >>> ex3 = exemplos_caravela()[2]
  >>> run_semantica(ex3)
  True
  >>> run_semantica(ex3.replace("pilar(3, azul)", "pilar(3, azl)"))
  'Erro semântico: Nome não declarado'
  >>> run_semantica(ex3.replace("pilar(3, azul)", "pilar(azul, 3)"))
  'Erro semântico: Argumentos do template com tipo errado'
  >>> run_semantica(ex3.replace("draw   templo, chao;", "draw   templo, chao;  if ang > 0 { }"))
  'Erro semântico: Nome não declarado'
  >>> run_semantica(ex3.replace("draw   templo, chao;", "draw   templo, cam;"))
  'Erro semântico: Tipo errado'
  >>> run_semantica(ex3.replace("lighting per_vertex; }", "lighting per_vertex { shininess 3; } }"))
  'Erro semântico: shininess só existe em lighting phong'

  Nomes e declarações globais:

  >>> run_semantica('let a = b; let b = 2; camera c { near a; }')
  True
  >>> run_semantica('let a = b + 1; let b = a;')
  'Erro semântico: Dependência circular entre let'
  >>> run_semantica('scene s = grid length 1 divisions 1; camera s { }')
  'Erro semântico: Nome declarado mais de uma vez'
  >>> run_semantica('let red = 1;')
  'Erro semântico: As cores pré-definidas não podem ser declaradas de novo'
  >>> run_semantica('template t(h: num) { u(h); } template u(h: num) { t(h); }')
  'Erro semântico: Template recursivo'

  Âmbitos locais:

  >>> run_semantica('scene s { let x = 1; box b { translate (x, 0, 0); } }')
  True
  >>> run_semantica('scene s { box b { translate (x, 0, 0); } let x = 1; }')
  'Erro semântico: Nome não declarado'
  >>> run_semantica('scene s { let x = 1; let x = 2; }')
  'Erro semântico: Nome declarado mais de uma vez'
  >>> run_semantica('scene s { if 1 < 2 { let x = 1; } box b { scale (x, x, x); } }')
  'Erro semântico: Nome não declarado'
  >>> run_semantica('template p(a: num, a: num) { }')
  'Erro semântico: Nome declarado mais de uma vez'

  Tipos:

  >>> run_semantica('let v = (1, 2, 3); let c = red; scene s { box b : m { translate v; } } material m { diffuse c; }')
  True
  >>> run_semantica('let v = 1; scene s { box b { translate v; } }')
  'Erro semântico: Tipo errado'
  >>> run_semantica('camera t { } material m { layer day t; }')
  'Erro semântico: Tipo errado'
  >>> run_semantica('template p(m: num) { box b : m; }')
  'Erro semântico: Tipo errado'

  Coerência dos materiais:

  >>> run_semantica('material m { lighting phong; lighting toon; }')
  'Erro semântico: Um material só pode ter um lighting'
  >>> run_semantica('texture t = "a.jpg"; material m { layer day t; layer day t; }')
  'Erro semântico: Camada repetida no material'
  >>> run_semantica('texture t = "a.jpg"; material m { layer day t; layer specular t; }')
  'Erro semântico: A camada specular exige lighting phong'
  >>> run_semantica('texture t = "a.jpg"; material m { lighting none; layer day t; layer night t; }')
  'Erro semântico: A camada night exige um modo com luz'
  >>> run_semantica('texture t = "a.jpg"; material m { lighting phong; layer specular t; }')
  'Erro semântico: As camadas specular, night e clouds exigem uma camada day'
  >>> run_semantica('pass p { lighting phong { levels 3; } for all; }')
  'Erro semântico: levels só existe em lighting toon'
  """
  try:
    analisar(data)
    return True
  except ContextConditionError as e:
    return f'Erro semântico: {e}'

doctest.run_docstring_examples(run_semantica, globals())

Por fim, os quatro exemplos da secção 1.6 estão semanticamente corretos. Para cada um mostra-se a tabela de símbolos global calculada (sem as cores pré-definidas) e as unidades de textura de cada material. No exemplo 4, as camadas `day`, `specular` e `night` recebem as unidades 0, 1 e 2. A camada `clouds` não recebe nenhuma, porque usa ruído e não uma textura.

In [16]:
for i, programa in enumerate(exemplos_caravela(), start=1):
  raiz = analisar(programa)
  print(f"Exemplo {i}:")
  print("  tabela:   ", {n: t for n, t in raiz.tabela.items() if n not in PRE})
  print("  unidades: ", raiz.materiais)

Exemplo 1:
  tabela:    {'teapot_l': 'scene', 'teapot_s': 'scene', 'al': 'scene', 'plane': 'scene', 'Yellow': 'material', 'Blue': 'material', 'MainCamera': 'camera', 'Sun': 'light', 'pass1': 'pass'}
  unidades:  {'Yellow': {}, 'Blue': {}}
Exemplo 2:
  tabela:    {'modo': 'num', 'rb': 'scene', 'al': 'scene', 'teapot_obj': 'scene', 'teapot_3ds': 'scene', 'plane': 'scene', 'MainCamera': 'camera', 'Sun': 'light', 'pass1': 'pass'}
  unidades:  {}
Exemplo 3:
  tabela:    {'n': 'num', 'raio': 'num', 'azul': 'material', 'branco': 'material', 'pilar': ('template', ('num', 'material')), 'templo': 'scene', 'chao': 'scene', 'cam': 'camera', 'sol': 'light', 'render': 'pass'}
  unidades:  {'azul': {}, 'branco': {}}
Exemplo 4:
  tabela:    {'earth': 'texture', 'spec': 'texture', 'eNight': 'texture', 'eClouds': 'texture', 'textured': 'material', 'Sphere': 'scene', 'MainCamera': 'camera', 'Sun': 'light', 'render': 'pass'}
  unidades:  {'textured': {'day': 0, 'specular': 1, 'night': 2}}


## 4. Qualidade da linguagem e da gramática

Nas aulas, a qualidade de uma gramática foi descrita por cinco propriedades: **precisão**, **legibilidade**, **manutenibilidade**, **extensibilidade** e **modularidade**. Estas propriedades podem entrar em conflito, e nem sempre é possível otimizá-las todas ao mesmo tempo. Esta secção avalia primeiro a linguagem (para quem a usa) e depois a gramática (para quem a mantém), com base em medidas calculadas sobre as próprias gramáticas. No fim, resume os compromissos que foram assumidos.

### 4.1 Medidas

As medidas são calculadas a partir das gramáticas das secções 2 e 3 e dos exemplos da secção 1.6. Assim, mantêm-se certas se as gramáticas mudarem. Primeiro apresentam-se contagens simples e, a seguir, as métricas de qualidade de gramáticas de Power e Malloy.

In [17]:
from lark import Tree

def medidas_gramatica(texto):
  g = Lark(texto, parser="lalr", lexer="basic")
  nao_terminais = {r.origin.name for r in g.rules if not r.origin.name.startswith("_")}
  reservadas = [t for t in g.terminals if t.pattern.type == "str" and re.fullmatch(r"[a-z_0-9]+", t.pattern.value)]
  return {"não-terminais": len(nao_terminais), "terminais": len(g.terminals), "palavras reservadas": len(reservadas),
          "linhas": len(texto.strip().splitlines())}

def profundidade(t):
  return 1 + max([profundidade(c) for c in t.children if isinstance(c, Tree)] + [0])

print("Gramática (secção 2):           ", medidas_gramatica(grammar))
print("Gramática do LarkAG (secção 3): ", medidas_gramatica(ag.getCFG()))
print("Gramática de atributos:         ", {
  "símbolos com atributos sintetizados": len(re.findall(r"^SA\(", ag_grammar, re.M)),
  "símbolos com atributos herdados": len(re.findall(r"^IA\(", ag_grammar, re.M)),
  "equações": len(re.findall(r"\b[a-z]+(?:\[\d\])?\.[a-z]+ = ", ag_grammar)),
  "condições de contexto": len(re.findall(r'-> "', ag_grammar)),
  "linhas": len(ag_grammar.strip().splitlines())})
for i, programa in enumerate(exemplos_caravela(), start=1):
  arvore = parser.parse(programa)
  print(f"Exemplo {i}: {len(programa.splitlines())} linhas, árvore com {len(list(arvore.iter_subtrees()))} nós e profundidade {profundidade(arvore)}")

Gramática (secção 2):            {'não-terminais': 71, 'terminais': 76, 'palavras reservadas': 58, 'linhas': 254}
Gramática do LarkAG (secção 3):  {'não-terminais': 72, 'terminais': 76, 'palavras reservadas': 58, 'linhas': 101}
Gramática de atributos:          {'símbolos com atributos sintetizados': 68, 'símbolos com atributos herdados': 6, 'equações': 222, 'condições de contexto': 25, 'linhas': 525}
Exemplo 1: 28 linhas, árvore com 124 nós e profundidade 11
Exemplo 2: 34 linhas, árvore com 155 nós e profundidade 15
Exemplo 3: 38 linhas, árvore com 267 nós e profundidade 18
Exemplo 4: 33 linhas, árvore com 112 nós e profundidade 11


Resultados (os números são os que a célula acima imprime):

| Medida | Valor |
|---|---|
| Não-terminais / terminais / palavras reservadas | 71 / 76 / 58 |
| Linhas da gramática da secção 2 | 254 |
| Gramática de atributos | 68 símbolos com atributos sintetizados, 6 com herdados, 222 equações, 25 condições de contexto, 525 linhas |
| Exemplos 1 a 4 | 28 a 38 linhas cada; árvores com 112 a 267 nós e profundidade 11 a 18 |
| Redução face aos ficheiros originais (secção 1.6) | 175 → 28, 191 → 34 e 232 (+132 de ruído) → 33 linhas: cerca de 6 vezes menos, num só ficheiro |

**Métricas de qualidade da gramática.** Para avaliar a gramática com valores que se podem comparar com os de outras gramáticas, usam-se as métricas de Power e Malloy (*A metrics suite for grammar-based software*, 2004). São calculadas sobre as regras tal como estão escritas (EBNF), e não sobre a versão BNF que o Lark gera internamente.

As métricas de **tamanho** medem quanto texto e quantas decisões tem a gramática. As de **estrutura** usam o grafo de dependências entre não-terminais, em que há uma aresta de `A` para `B` se `B` aparece no lado direito de `A`. Um **nível** é um conjunto de não-terminais que se usam uns aos outros (recursividade mútua); um não-terminal que não é mutuamente recursivo forma um nível sozinho.

| Métrica | Definição | O que indica |
|---|---|---|
| TERM | número de terminais | tamanho do vocabulário |
| VAR | número de não-terminais | tamanho da gramática |
| MCC | complexidade de McCabe: número de pontos de decisão (`\|`, `?`, `*`) | quantas alternativas o *parser* e quem lê têm de considerar |
| AVS | tamanho médio do lado direito (símbolos por não-terminal) | regras curtas são mais fáceis de ler |
| HAL | esforço de Halstead, em milhares, com os símbolos como operandos e os metassímbolos (`:`, `\|`, `?`, `*`, parênteses) como operadores | esforço para compreender a gramática |
| TIMP | impureza de árvore: `2(e − n + 1) / ((n − 1)(n − 2))`, com `n` não-terminais e `e` arestas | 0% se o grafo é uma árvore, 100% se todos dependem de todos |
| CLEV | número de níveis / VAR | perto de 100%: poucas regras mutuamente recursivas, gramática fácil de dividir em partes |
| NSLEV | número de níveis com mais de um não-terminal | quantos grupos de recursividade mútua existem |
| DEP | tamanho do maior nível | o maior grupo de regras que tem de ser percebido em conjunto |
| HEI | altura: o caminho mais comprido de níveis a partir de `start` | quantas camadas de regras separam o programa de um valor |

Acrescentam-se duas medidas do *parser* gerado: o número de produções depois de o Lark converter a gramática para BNF e o número de estados da tabela LALR(1).

Para a gramática de atributos, conta-se cada equação pelo que faz: **cópia** (passa um atributo de outro símbolo sem o alterar, como `diffuse.usos = colorval.usos`), **constante** (`[]`, `None`, `K_PHONG`, ...) ou **calculada** (todas as outras).

In [18]:
import math
from lark.load_grammar import GrammarBuilder

def regras_ebnf(texto):
  """As regras tal como estão escritas (EBNF): nome -> árvore do lado direito."""
  gb = GrammarBuilder()
  gb.load_grammar(texto)
  return {str(n): d.tree for n, d in gb._definitions.items() if not d.is_term}

def simbolos(arvore):
  """Os símbolos (operandos) e os metassímbolos (operadores) de um lado direito."""
  operandos, operadores = [], []
  for t in arvore.iter_subtrees_topdown():
    if t.data == "expansions" and len(t.children) > 1:
      operadores += ["|"] * (len(t.children) - 1)
    elif t.data == "expansions" and t is not arvore:
      operadores.append("( )")
    elif t.data == "expr":
      operadores.append(str(t.children[1]))
    elif t.data == "value":
      s = t.children[0]
      operandos.append(str(s.name) if hasattr(s, "name") else str(s.children[0]))
  return operandos, operadores

def alcancaveis(grafo, a):
  vistos, pilha = set(), [a]
  while pilha:
    for b in grafo[pilha.pop()]:
      if b not in vistos:
        vistos.add(b)
        pilha.append(b)
  return vistos

def metricas_gramatica(texto, inicial="start"):
  """Métricas de tamanho e de estrutura de Power e Malloy (2004)."""
  regras = regras_ebnf(texto)
  g = Lark(texto, parser="lalr", lexer="basic")

  # tamanho
  operandos, operadores = [], []
  for nome, arvore in regras.items():
    o, m = simbolos(arvore)
    operandos += o
    operadores += [":"] + m
  decisoes = sum(1 for m in operadores if m in ("|", "?", "*", "+"))
  n1, n2 = len(set(operadores)), len(set(operandos))
  N1, N2 = len(operadores), len(operandos)
  volume = (N1 + N2) * math.log2(n1 + n2)
  dificuldade = n1 / 2 * N2 / n2

  # estrutura: grafo "A usa B", e os níveis (componentes fortemente ligadas)
  grafo = {a: {s for s in simbolos(t)[0] if s in regras} for a, t in regras.items()}
  alc = {a: alcancaveis(grafo, a) for a in regras}
  nivel = {a: frozenset({a} | {b for b in alc[a] if a in alc[b]}) for a in regras}
  niveis = set(nivel.values())
  arestas = {frozenset((a, b)) for a in grafo for b in grafo[a] if a != b}
  n, e = len(regras), len(arestas)

  def altura(lv, memo={}):
    if lv not in memo:
      seguintes = {nivel[b] for a in lv for b in grafo[a]} - {lv}
      memo[lv] = 1 + max([altura(s) for s in seguintes] + [0])
    return memo[lv]

  return {
    "TERM": len(g.terminals),
    "VAR": n,
    "MCC": decisoes,
    "AVS": round(N2 / n, 2),
    "HAL": round(dificuldade * volume / 1000, 1),
    "TIMP": round(200 * (e - n + 1) / ((n - 1) * (n - 2)), 1),
    "CLEV": round(100 * len(niveis) / n, 1),
    "NSLEV": sum(1 for lv in niveis if len(lv) > 1),
    "DEP": max(len(lv) for lv in niveis),
    "HEI": altura(nivel[inicial], {}),
    "produções BNF": len(g.rules),
    "estados LALR": len(g.parser.parser.parser.parse_table.states),
    "maior nível": sorted(max(niveis, key=len)),
  }

def metricas_atributos(texto):
  """Equações da gramática de atributos, separadas pelo que fazem."""
  lados = re.findall(r"\b[a-z]+(?:\[\d\])?\.[a-z]+ = ([^;]*);", texto)
  copia = [r for r in lados if re.fullmatch(r"[a-z]+(?:\[\d\])?\.[a-z]+", r.strip())]
  constante = [r for r in lados if re.fullmatch(r"\[\]|\{\}|None|True|False|K_[A-Z]+", r.strip())]
  sa = re.findall(r"^SA\(\w+\)\s*\{([^}]*)\}", texto, re.M)
  ia = re.findall(r"^IA\(\w+\)\s*\{([^}]*)\}", texto, re.M)
  return {
    "atributos sintetizados": sum(s.count(":") for s in sa),
    "atributos herdados": sum(s.count(":") for s in ia),
    "equações": len(lados),
    "cópias": len(copia),
    "constantes": len(constante),
    "calculadas": len(lados) - len(copia) - len(constante),
    "condições de contexto": len(re.findall(r'-> "', texto)),
  }

for titulo, texto in [("Gramática (secção 2)", grammar), ("Gramática do LarkAG (secção 3)", ag.getCFG())]:
  print(titulo)
  for k, v in metricas_gramatica(texto).items():
    print(f"  {k:<14} {v}")
print("Gramática de atributos")
for k, v in metricas_atributos(ag_grammar).items():
  print(f"  {k:<23} {v}")

Gramática (secção 2)
  TERM           76
  VAR            71
  MCC            83
  AVS            4.11
  HAL            16.3
  TIMP           1.4
  CLEV           90.1
  NSLEV          3
  DEP            4
  HEI            11
  produções BNF  174
  estados LALR   330
  maior nível    ['scene_block', 'scene_for', 'scene_if', 'scene_item']
Gramática do LarkAG (secção 3)
  TERM           76
  VAR            72
  MCC            80
  AVS            4.58
  HAL            15.7
  TIMP           1.4
  CLEV           88.9
  NSLEV          3
  DEP            5
  HEI            11
  produções BNF  169
  estados LALR   324
  maior nível    ['sceneblock', 'scenefor', 'sceneif', 'sceneitem', 'sceneitems']
Gramática de atributos
  atributos sintetizados  102
  atributos herdados      6
  equações                222
  cópias                  89
  constantes              41
  calculadas              92
  condições de contexto   25


Resultados:

| Métrica | Gramática (secção 2) | Gramática do LarkAG (secção 3) |
|---|---|---|
| TERM | 76 | 76 |
| VAR | 71 | 72 |
| MCC | 83 | 80 |
| AVS | 4,11 | 4,58 |
| HAL (milhares) | 16,3 | 15,7 |
| TIMP | 1,4% | 1,4% |
| CLEV | 90,1% | 88,9% |
| NSLEV | 3 | 3 |
| DEP | 4 | 5 |
| HEI | 11 | 11 |
| Produções BNF | 174 | 169 |
| Estados LALR(1) | 330 | 324 |

| Gramática de atributos | Valor |
|---|---|
| Atributos | 102 sintetizados e 6 herdados |
| Equações | 222: 89 cópias (40%), 41 constantes (18%) e 92 calculadas (41%) |
| Condições de contexto | 25 |

**O que os valores mostram.**

- **A estrutura é quase uma árvore.** TIMP = 1,4% e CLEV = 90%: 64 dos 71 não-terminais formam um nível sozinhos, ou seja, quase todas as regras só dependem de regras "mais abaixo". Isto confirma a organização por grupos da gramática (secção 2.3): cada grupo pode ser lido e alterado sem conhecer os outros.
- **A recursividade mútua está só onde a linguagem a pede.** Há apenas 3 níveis com mais de um não-terminal (NSLEV = 3): os itens das cenas (`scene_item`, `scene_for`, `scene_if`, `scene_block`, o maior, com DEP = 4), os itens dos passes (`pass_item`, `pass_if`, `pass_block`) e as expressões (`num`, `term`, `factor`). São exatamente as construções aninhadas da secção 1.5. As listas (`params`, `args`, `id_list`) são recursivas, mas só sobre si próprias.
- **As regras são curtas, mas são muitas.** AVS ≈ 4: uma regra típica é `near: "near" num ";"`. Em troca, VAR = 71, porque cada propriedade tem a sua regra (decisão da secção 2.3.1).
- **A gramática é funda.** HEI = 11: o caminho mais comprido é `start → decl → scene_decl → scene_item → object → object_body → transform → translate → vec3_val → vec3 → num`. Com a recursividade dos blocos e de `num / term / factor`, isto dá as árvores com profundidade até 18 dos exemplos.
- **As duas versões da gramática têm a mesma estrutura.** Os valores da gramática do LarkAG quase não mudam. As diferenças vêm das alterações da secção 3.5, como as listas recursivas à esquerda (`sceneitems` passa a fazer parte do maior nível, DEP = 5).
- **O *parser* gerado tem tamanho moderado:** 330 estados LALR(1), sem conflitos.
- **40% das equações da gramática de atributos são cópias.** 89 das 222 equações só passam um atributo de um filho para o pai, quase sempre `usos`. É o custo de recolher os usos de baixo para cima (secção 3.2): cada regra intermédia tem de os copiar. Num sistema de gramáticas de atributos com regras de cópia implícitas, estas equações não precisavam de ser escritas.

### 4.2 Qualidade da linguagem

**Adequação ao domínio e concisão.** Os conceitos da linguagem (`texture`, `material`, `camera`, `light`, `pass`, `layer`) são os mesmos que os alunos já usam em VTR, e os valores escrevem-se como no domínio: vetores `(x, y, z)` e cores `rgb(...)` ou pelo nome. Os três projetos das aulas ficam cerca de 6 vezes mais curtos, num único ficheiro em vez de quatro. O utilizador deixa de escrever GLSL e nomes de uniforms.

**Legibilidade.** Cada coisa declara-se uma vez, num bloco com um nome, e lê-se de cima para baixo (texturas, materiais, cenas, câmara, luz, passe). Os modos de iluminação e as camadas dizem *o que* se quer ver (`lighting phong`, `layer night`), e não *como* se calcula.

**Fiabilidade (deteção de erros).** É a melhoria principal em relação ao XML do Nau, onde um nome errado não dá nenhum aviso: o objeto aparece preto. Na Caravela, os erros são apanhados em três níveis:

| Nível | Exemplos de erros apanhados |
|---|---|
| Léxico | `texture` não pode ser um nome; `drawcam` não é lido como `draw cam` |
| Sintaxe | `position 5;`, `diffuse 3;`, `draw` dentro de um material, vetores com 2 componentes |
| Semântica (secção 3) | nomes não declarados ou repetidos, `draw cam`, `pilar(azul, 3)`, `let` em ciclo, `shininess` fora de `phong`, camadas incompatíveis |

**Limitações.**

- **Mensagens de erro pouco precisas.** No LarkAG, a mensagem de uma condição de contexto é um texto fixo. Por isso o erro diz `Nome não declarado`, mas não diz qual nome nem em que linha. Para os nomes globais isto é ainda mais visível, porque a verificação é feita na raiz, longe do sítio onde o nome foi usado.
- **Expressividade reduzida, de propósito.** As condições têm uma só comparação (não há `and`, `or` nem `not`), o `for` só avança de 1 em 1, e não há funções como `sin` ou `cos`. Os efeitos disponíveis são só os das aulas 2 e 3 (secção 1.3).
- **58 palavras reservadas.** Palavras do domínio como `texture`, `scale` ou `position` não podem ser usadas como nomes. No exemplo 4, o passe que no original se chama `texture` teve de passar a `render`.

### 4.3 Qualidade da gramática

| Propriedade | Pontos fortes | Pontos fracos |
|---|---|---|
| **Precisão** | A gramática é LALR(1) e não é ambígua (verificado com `strict=True`; 330 estados, sem conflitos). Cada propriedade aceita só o tipo de valor certo (`position vec3_val`, `near num`), e por isso muitos erros são de sintaxe. Os doctests e os exemplos são testados automaticamente, e os exemplos são lidos do próprio relatório. | Um `ID` sozinho num valor é lido como `num` (`pilar(3, azul)`), e o seu tipo só fica certo na semântica, com o atributo `nome` (secção 3.3). |
| **Legibilidade** | A gramática está organizada pela ordem dos conceitos da secção 1.4 e usa a notação das aulas (`expr`/`term`/`factor`, recursividade à esquerda, terminais com nome). Os nomes das regras são os nomes dos conceitos, e as regras são curtas (AVS ≈ 4 símbolos). | A gramática é funda (HEI = 11) e as árvores ainda mais: até 18 níveis nos exemplos, porque cada número passa por `num → term → factor`, e cada propriedade tem a sua própria regra (`position`, `near`, ...). O `?` do Lark, que esconde nós com um só filho, foi evitado para manter a forma das aulas e para o LarkAG ter um nó em cada nível. |
| **Manutenibilidade** | Cada decisão está justificada (secção 2.3.1). Os testes falham se a gramática deixar de aceitar os exemplos. | A gramática existe em duas versões, a da secção 2 e a do LarkAG (secção 3.5), que têm de se manter iguais. O teste de 3.7 apanha as diferenças, mas não as evita. A gramática de atributos é longa (525 linhas), sobretudo por causa das regras que só copiam `usos` de um filho para o pai (89 das 222 equações, 40%). |
| **Extensibilidade** | Uma cor nova não exige mudar a gramática: basta acrescentá-la ao ambiente inicial (`PRE`). Um modo de iluminação ou um papel de camada novo é um terminal, uma alternativa e, se for preciso, uma condição de contexto. Um tipo de propriedade novo numa câmara é uma alternativa de `camera_prop`. | Um tipo de valor novo (por exemplo, `vec2`) obriga a mudar `value`, os tipos dos parâmetros e a função `compativel`. |
| **Modularidade** | A gramática está separada em grupos (programa, materiais, cenas, câmara e luz, passes, valores, léxico). O grafo de dependências é quase uma árvore (TIMP = 1,4%, CLEV = 90%), e só há 3 grupos de regras mutuamente recursivas, o maior com 4 regras (NSLEV = 3, DEP = 4). Há um `if` por contexto, para que cada grupo só aceite os seus próprios itens. A semântica está separada em atributos com funções diferentes (usos, ambiente local, tabela global, coerência do material). | O Lark e o LarkAG não têm módulos: a gramática é um único texto. Quase todos os símbolos sintetizam `usos`, o que liga a semântica de nomes a quase todas as regras. |

### 4.4 Compromissos assumidos

| Decisão | O que se ganha | O que se perde |
|---|---|---|
| Valores com tipo na sintaxe (`vec3_val`, `color_val`, `num`) | mais erros apanhados pelo *parser*, com a posição exata | mais regras; o tipo de um `ID` sozinho continua a ser uma questão da semântica |
| Um `if` por contexto (`scene_if`, `pass_if`) | `draw` numa cena é um erro de sintaxe | duas regras quase iguais, e o dobro das regras na gramática de atributos |
| Sem `if` dentro de materiais | regras de coerência e numeração das unidades simples (um material tem sempre as mesmas camadas) | para mudar o modo de iluminação usa-se o `pass_if` (exemplo 2), e não o material |
| Cores pré-definidas como `ID` | menos 7 palavras reservadas; cores novas sem mudar a gramática | `background verde;` passa na sintaxe e só falha na semântica |
| `lexer="basic"` com palavras reservadas | separação clara entre palavras reservadas e nomes | 58 palavras que não se podem usar como nomes |
| Nomes globais usados antes da declaração | a ordem das declarações é livre, como no XML do Nau | os usos globais só são verificados na raiz, com mensagens menos precisas |
| Listas recursivas à esquerda na gramática de atributos | o `env` passa de item para item, como nas aulas | a gramática do LarkAG fica diferente da secção 2 (secção 3.5) |

### 4.5 Melhorias possíveis

- **Mensagens com nome e posição.** Guardar em cada uso a linha do `ID` (o Lark tem essa informação nos tokens) e verificar os nomes com uma função que lance o erro com uma mensagem construída, em vez do texto fixo da condição de contexto.
- **Valores válidos.** Acrescentar um atributo `val` que avalia as expressões, como no exemplo das aulas, para verificar `near > 0`, `far > near`, `levels >= 2` e as componentes das cores.
- **Condições compostas.** Acrescentar `and`, `or` e `not` a `cond`, com os níveis de precedência habituais.
- **Gramática única.** Escrever a gramática da secção 2 já com nomes sem `_` e listas recursivas, para haver uma só versão. Em troca, a gramática fica menos próxima da notação EBNF.

## 5. Conclusão

Neste trabalho desenhámos a Caravela, uma linguagem para descrever cenas 3D do Nau sem ter de escrever XML nem GLSL. Fizemos as quatro tarefas do enunciado:

1. **Desenho da linguagem** (secção 1): partimos de um problema que conhecemos de VTR e reescrevemos três projetos das aulas em Caravela, com cerca de 6 vezes menos linhas e num só ficheiro.
2. **Léxico e sintaxe** (secção 2): gramática em Lark, LALR(1) e sem conflitos, testada com doctests e com os exemplos do próprio relatório.
3. **Semântica** (secção 3): gramática de atributos em LarkAG que verifica nomes, âmbitos, tipos, templates e a coerência dos materiais, e que calcula as unidades de textura.
4. **Qualidade** (secção 4): avaliámos a linguagem e a gramática com as propriedades e as métricas dadas nas aulas.

A parte mais delicada foi tratar os nomes globais na gramática de atributos. Como podem ser usados antes de serem declarados, não dava para os passar para baixo num `env`, e por isso fazemo-los subir até à raiz (secção 3.2). Funciona, mas as mensagens de erro ficam pouco precisas, e é a primeira coisa que melhorávamos (secção 4.5).

Na Parte II vamos reutilizar a Caravela para representar, analisar, visualizar e transformar programas, como pede o enunciado. O gerador dos ficheiros do Nau descrito no Anexo B encaixa nessa parte.

## Anexos

### Anexo A — Ficheiros originais dos exemplos

Os ficheiros originais dos projetos das aulas de VTR reescritos em Caravela na secção 1.6. Cada ficheiro pode ser expandido para comparar com o programa em Caravela correspondente.

#### A.1 Exemplo 1: iluminação simples (aula 2)

<details>
<summary><b>Original:</b> <code>simple.xml</code> (67 linhas)</summary>

```xml
<project name="Simple" >
    <assets>
        <scenes>
            <scene name="teapot_l" param="SWAP_YZ UNITIZE">
                <file name="models/teapot.3ds"/>
            </scene>

            <scene name="teapot_s" param="UNITIZE">
                <file name="models/teapot.obj"/>
            </scene>

            <scene name="al" param="UNITIZE">
                <file name="models/al.obj"/>
            </scene>
            
            <scene name="plane" >
                <geometry name = "Grid" type = "GRID" LENGTH="20"  DIVISIONS="2"></geometry>
            </scene>
        </scenes>
        
        <viewports>
            <viewport name="MainViewport">
                <CLEAR_COLOR r="1.0" g="1.0" b="1.0" />
            </viewport>
        </viewports>        

        <cameras>
            <camera name="MainCamera" >
                <viewport name="MainViewport" />
                <POSITION x="1.46" y="1.05" z="1.75"  />
                <LOOK_AT_POINT x = "0" y="0" z="0" />
                <NEAR value = "0.01" />
            </camera>
        </cameras>
        
        <lights>
            <light name="Sun">
                <DIRECTION x="2" y="1" z="2" />
            </light>
        </lights>

        <materialLibs>
            <materialLib filename="simple.mlib"/>
        </materialLibs>

    </assets>
    
    <pipelines>
        <pipeline name="simple" default="true">
            <pass class="default" name="pass1">
                <scenes>
                    <scene name="teapot_s" />
                </scenes>
                <camera name="MainCamera" />
                <lights>
                    <light name="Sun" />
                </lights>
                <injectionMaps>
                    <map toMaterial="*">
                        <shader fromLibrary="simple" fromMaterial="simple" />
                    </map>
                </injectionMaps>
            </pass>
        </pipeline>
    </pipelines>

</project>
```

</details>

<details>
<summary><b>Original:</b> <code>simple.mlib</code> (73 linhas)</summary>

```xml
<?xml version="1.0" ?>
<materialLib name="simple">

    <shaders>
        <shader name="simple" vs="shaders/simple.vert" ps="shaders/simple.frag" />
        <shader name="v4" vs="shaders/v4.vert" ps="shaders/v4.frag" />

    </shaders>

    <materials>

        <material name="Yellow">
            <color>
                <DIFFUSE r="1.0" g="0.87" b="0.0" />
            </color>
        </material>

        <material name="Blue">
            <color>
                <DIFFUSE r="0.0" g="0.34" b="0.72" />
            </color>
        </material>


        <material name="simple">
            <shader name="simple" >
                <values>
                    <valueof uniform="m_pvm" type="RENDERER" component="PROJECTION_VIEW_MODEL" />
                    <valueof uniform="m_view" type="RENDERER" component="VIEW" />
                    <valueof uniform="m_normal" type="RENDERER" component="NORMAL" />
                    <valueof uniform="l_dir" type="LIGHT" component="DIRECTION" id="0" />
                    <valueof uniform="diffuse" type="COLOR" component="DIFFUSE" />
                </values>
            </shader>
        </material>

        <material name="v4">
            <color>
                <DIFFUSE r="1.0" g="0.87" b="0.0" a="1.0" />
                <SHININESS value="100" />
            </color>
            <shader name="v4" >
                <values>
                    <valueof uniform="m_pvm"
                            type="RENDERER"
                            component="PROJECTION_VIEW_MODEL" />
                    <valueof uniform="m_vm"
                            type="RENDERER"
                            component="VIEW_MODEL" />
                    <valueof uniform="m_v"
                            type="RENDERER"
                            component="VIEW" />
                    <valueof uniform="m_n"
                            type="RENDERER"
                            component="NORMAL" />
                    <valueof uniform="ld"
                            type="LIGHT"
                            component="DIRECTION" id="0" />
                    
                    <valueof uniform="diffuse"
                            type="COLOR"
                            component="DIFFUSE" />
                    <valueof uniform="shininess"
                            type="COLOR"
                            component="SHININESS" />
                </values>
            </shader>       
        </material>


    </materials>

</materialLib>
```

</details>

<details>
<summary><b>Original:</b> <code>shaders/simple.vert</code> (25 linhas)</summary>

```glsl
#version 460

uniform mat4 m_pvm, m_view;
uniform mat3 m_normal;
uniform vec4 diffuse;

uniform vec4 l_dir;

in vec4 position;
in vec3 normal;

out vec4 color_V;

void main() {

    vec3 n = normalize(m_normal * normal);
    vec4 l4 = m_view * l_dir;
    vec3 l = normalize(vec3(l4));

    float intensity = max(0.0, dot(l,n));

    color_V = intensity * diffuse; 

    gl_Position = m_pvm * position;
}
```

</details>

<details>
<summary><b>Original:</b> <code>shaders/simple.frag</code> (10 linhas)</summary>

```glsl
#version 460

in vec4 color_V;

out vec4 color;

void main() {

    color = color_V;
}
```

</details>

#### A.2 Exemplo 2: iluminação por píxel (aula 2)

O `.mlib` deste projeto é o mesmo do exemplo 1 (`simple.mlib`, no Anexo A.1), onde está o material `v4` com `shininess` 100.

<details>
<summary><b>Original:</b> <code>simple4.xml</code> (72 linhas)</summary>

```xml

<project name="Simple" >
    <assets>
        <scenes>
            <scene name="rb" param="UNITIZE">
                <file name="models/rainbowBench.obj"/>
            </scene>
            <scene name="al" param="UNITIZE">
                <file name="models/al.obj"/>
            </scene>
            <scene name="teapot_obj" param="UNITIZE">
                <file name="models/teapot.obj"/>
            </scene>
            <scene name="teapot_3ds" param="UNITIZE SWAP_YZ">
                <file name="models/teapot.3ds"/>
            </scene>
            <scene name="plane" >
                <geometry name = "Grid" type = "GRID" LENGTH="20"  DIVISIONS="2"></geometry>
            </scene>

        </scenes>
        
        <viewports>
            <viewport name="MainViewport">
                <CLEAR_COLOR r="0.15" g="0.25" b="0.45" />
            </viewport>
        </viewports>        

        <cameras>
            <camera name="MainCamera" >
                <viewport name="MainViewport" />
                <POSITION x="1.46" y="1.05" z="1.75"  />
                <LOOK_AT_POINT x = "0" y="0" z="0" />
                <NEAR value="0.1" />
            </camera>
        </cameras>
        
        <lights>
            <light name="Sun">
                <DIRECTION x="-2" y="-1" z="-2" />
            </light>
        </lights>

        <!-- adding a material lib -->
        <materialLibs>
            <materialLib filename="simple.mlib"/>
        </materialLibs>


    </assets>
    
    <pipelines>
        <pipeline name="simple" default="true">
            <pass class="default" name="pass1">
                <scenes>
                    <scene name="teapot_3ds" />
                </scenes>
                <camera name="MainCamera" />
                <lights>
                    <light name="Sun" />
                </lights>
                <injectionMaps>
                    <map toMaterial="*">
                        <shader fromLibrary="simple" fromMaterial="v4" />
                    </map>   
                </injectionMaps>
            </pass>
        </pipeline>
    </pipelines>


</project>
```

</details>

<details>
<summary><b>Original:</b> <code>shaders/v4.vert</code> (21 linhas)</summary>

```glsl
#version 460

in vec4 position; // local
in vec3 normal;   // local

uniform mat4 m_pvm, m_vm, m_v;
uniform mat3 m_n;
uniform vec4 ld;  // direcao da luz (espaco global)

out vec3 nn;      // normal (espaco camara)
out vec3 ldn;     // direcao para a luz (espaco camara)
out vec3 e;       // vetor para o olho (espaco camara)

void main() {

    nn  = normalize(m_n * normal);
    ldn = normalize(vec3(m_v * -ld));
    e   = -vec3(m_vm * position);

    gl_Position = m_pvm * position;
}
```

</details>

<details>
<summary><b>Original:</b> <code>shaders/v4.frag</code> (25 linhas)</summary>

```glsl
#version 460

in vec3 nn, ldn, e;

uniform vec4 diffuse;
uniform float shininess;

out vec4 color;

void main() {

    vec3 n = normalize(nn);
    vec3 l = normalize(ldn);

    float intensity = max(0.0, dot(n, l));
    vec4 specular = vec4(0.0);

    if (intensity > 0.0) {
        vec3 h = normalize(normalize(e) + l);
        float spec = max(0.0, dot(h, n));
        specular = vec4(1.0) * pow(spec, shininess);
    }

    color = max(0.25, intensity) * diffuse + specular;
}
```

</details>

#### A.3 Exemplo 4: texturas da Terra (aula 3)

<details>
<summary><b>Original:</b> <code>textureModel.xml</code> (58 linhas)</summary>

```xml
<?xml version="1.0" ?>
<project name="Texture Model" >
    <assets>
    
        <scenes>
            <scene name="Sphere" param="UNITIZE">
                <file name="models/Sphere.obj"/>
            </scene>
        </scenes>
        
        <viewports>
            <viewport name="MainViewport">
                <CLEAR_COLOR r="0.0" g="0.0" b="0.0" />
            </viewport>
        </viewports>
        
        <cameras>
            <camera name="MainCamera" >
                <viewport name="MainViewport" />
                <POSITION x="0" y="0" z="3" />
                <VIEW x="0" y="0" z="-1.0" />
                <NEAR value="0.1" />
            </camera>
        </cameras>
        
        <lights>
            <light name="Sun">
                <DIRECTION x="2" y="-1" z="-2" />
            </light>
        </lights>
        
        <materialLibs>
            <materialLib filename="textureModel.mlib" />
        </materialLibs>
        
    </assets>
    
    <pipelines>
    
        <pipeline name="textures" >
        
            <pass class="default" name="texture" >
                <scenes>
                    <scene name="Sphere" />
                </scenes>
                <camera name="MainCamera" />
                <lights>
                    <light name="Sun" />
                </lights>
                <materialMaps>
                    <map fromMaterial="*" toMaterial="textured" toLibrary="Texture" />
                </materialMaps>
                
            </pass>
        </pipeline>
        
    </pipelines>
</project>
```

</details>

<details>
<summary><b>Original:</b> <code>textureModel.mlib</code> (76 linhas)</summary>

```xml
<?xml version="1.0" ?>
<materialLib name="Texture">

    <textures>
        <texture name="earth" filename="otherTextures/earthDay.jpg" mipmap="true" />
        <texture name="spec" filename="otherTextures/earthSpec.jpg" mipmap="true" />
        <texture name="eNight" filename="otherTextures/earthLights.jpg" mipmap="true" />
        <texture name="eClouds" filename="otherTextures/earthClouds.jpg" mipmap="true" />
    </textures>

    <shaders>

        <shader name="textured">
            <vs>
                <file name = "shaders/texture.vert" />
            </vs>
            <ps>
                <file name = "shaders/texture.frag" />
                <file name = "shaders/noise.frag" />
            </ps>
        </shader>
    </shaders>                  
    
    <materials>
        <material name="textured">
            <shader name="textured">
                <values>
                    <valueof uniform="m_pvm" 
                             type="RENDERER" 
                             component="PROJECTION_VIEW_MODEL" />
                     
                    <valueof uniform="m_viewModel" 
                             type="RENDERER"  
                             component="VIEW_MODEL" />
                     
                    <valueof uniform="m_view" 
                             type="RENDERER" 
                             component="VIEW" />
                     
                    <valueof uniform="m_normal" 
                             type="RENDERER" 
                             component="NORMAL" />
                     
                    <valueof uniform="l_dir"
                             type="LIGHT" 
                             component="DIRECTION" id="0"/>
                    <valueof uniform="time"
                             type="RENDERER" 
                             component="TIMER" />

                    <valueof uniform="texEarth"
                             type="TEXTURE_BINDING" context="CURRENT"
                             component="UNIT" id="0" />
                    <valueof uniform="textSpec"
                             type="TEXTURE_BINDING" context="CURRENT"
                             component="UNIT" id="1" />
                    <valueof uniform="texNight"
                             type="TEXTURE_BINDING" context="CURRENT"
                             component="UNIT" id="2" />
                    <valueof uniform="texClouds"
                             type="TEXTURE_BINDING" context="CURRENT"
                             component="UNIT" id="3" />
                    

                </values>
            </shader>
            <textures>
                <texture name="earth" UNIT="0" />
                <texture name="spec" UNIT="1" />
                <texture name="eNight" UNIT="2" />
                <texture name="eClouds" UNIT="3" />
            </textures>
        </material>
    </materials>
    
</materialLib>
```

</details>

<details>
<summary><b>Original:</b> <code>shaders/texture.vert</code> (28 linhas)</summary>

```glsl
#version 330

uniform mat4 m_pvm;
uniform mat4 m_viewModel;
uniform mat4 m_view;
uniform mat3 m_normal;

uniform vec4 l_dir;    // global space

in vec4 position;   // local space
in vec3 normal;     // local space
in vec2 texCoord0;

// the data to be sent to the fragment shader

out vec4 eye;
out vec3 n;
out vec3 ld;
out vec2 textCoord;

void main () {
    textCoord = texCoord0;
    n = normalize(m_normal * normal);
    eye = -(m_viewModel * position);
    ld = normalize(vec3(m_view * -l_dir));

    gl_Position = m_pvm * position; 
}
```

</details>

<details>
<summary><b>Original:</b> <code>shaders/texture.frag</code> (70 linhas)</summary>

```glsl
#version 330

uniform float shininess = 128;
uniform sampler2D texEarth;
uniform sampler2D textSpec;
uniform sampler2D texNight;
uniform sampler2D texClouds;
uniform float time;

in  vec4 eye;
in  vec3 n;
in  vec3 ld;
in vec2 textCoord;
out vec4 colorOut;

float snoise(vec4);

float perlinNoise(vec4 p){
    float c = 0.0, amp = 1.0;
    float freq = 1.0;

    for(int i = 0; i < 5; ++i){
        c = c + snoise(p * freq) * amp;
        amp  *= 0.5;
        freq *= 2.0;
    }
    return c * 0.5 + 0.5;
}

const float PI = 3.14159;

void main() {

    vec4 eColor = texture(texEarth, textCoord);
    vec4 eSpec = texture(textSpec, textCoord);
    vec4 eNight = texture(texNight, textCoord);
    //vec4 eclouds = texture(texClouds, textCoord);
    // set the specular term to black
    vec2 t = vec2(2 * PI * textCoord.s + time * 0.00002, PI * (textCoord - 0.5) );
    vec3 sc = vec3(sin(t.s) * cos(t.t) , sin(t.t), cos(t.s)*cos(t.t));

    float eclouds = perlinNoise(vec4(textCoord * 32.0, time * 0.0001, 0.0));
    eclouds = smoothstep(0.4,0.9,eclouds);
    vec4 spec = vec4(0.0);

    // normalize both input vectors
    vec3 nd = normalize(n);
    vec3 e = normalize(vec3(eye));

    float intensity = max(dot(nd,ld), 0.0);

    // if the vertex is lit compute the specular color
    if (intensity > 0.0) {
        // compute the half vector
        vec3 h = normalize(ld + e); 
        // compute the specular intensity
        float intSpec = max(dot(h,nd), 0.0);
        // compute the specular term into spec
        // the spec map is white on the oceans and black on land
        // earthSpec.jpg has a single channel, so only .r holds the value
        spec = vec4(eSpec.r) * pow(intSpec,shininess);
    }
    vec4 colorDay = max(intensity * eColor + spec, eColor * 0.25) * (1.0 - eclouds) + eclouds;
    vec4 colorNight = eNight * (1.0 - eclouds);

    float f = smoothstep(0.0, 0.1, intensity);
    colorOut = mix(colorNight, colorDay,f);


}
```

</details>

<details>
<summary><b>Original:</b> <code>shaders/noise.frag</code> (132 linhas)</summary>

Biblioteca externa de ruído (Ashima Arts, licença MIT), usada pelo `texture.frag`. Não é escrita pelos alunos: o compilador da Caravela copia-a sem alterações quando um material usa `layer clouds noise`.

```glsl
//
// Description : Array and textureless GLSL 2D/3D/4D simplex 
//               noise functions.
//      Author : Ian McEwan, Ashima Arts.
//  Maintainer : stegu
//     Lastmod : 20110822 (ijm)
//     License : Copyright (C) 2011 Ashima Arts. All rights reserved.
//               Distributed under the MIT License. See LICENSE file.
//               https://github.com/ashima/webgl-noise
//               https://github.com/stegu/webgl-noise
// 

vec4 mod289(vec4 x) {
  return x - floor(x * (1.0 / 289.0)) * 289.0; }

float mod289(float x) {
  return x - floor(x * (1.0 / 289.0)) * 289.0; }

vec4 permute(vec4 x) {
     return mod289(((x*34.0)+10.0)*x);
}

float permute(float x) {
     return mod289(((x*34.0)+10.0)*x);
}

vec4 taylorInvSqrt(vec4 r)
{
  return 1.79284291400159 - 0.85373472095314 * r;
}

float taylorInvSqrt(float r)
{
  return 1.79284291400159 - 0.85373472095314 * r;
}

vec4 grad4(float j, vec4 ip)
  {
  const vec4 ones = vec4(1.0, 1.0, 1.0, -1.0);
  vec4 p,s;

  p.xyz = floor( fract (vec3(j) * ip.xyz) * 7.0) * ip.z - 1.0;
  p.w = 1.5 - dot(abs(p.xyz), ones.xyz);
  s = vec4(lessThan(p, vec4(0.0)));
  p.xyz = p.xyz + (s.xyz*2.0 - 1.0) * s.www; 

  return p;
  }
                        
// (sqrt(5) - 1)/4 = F4, used once below
#define F4 0.309016994374947451

float snoise(vec4 v)
  {
  const vec4  C = vec4( 0.138196601125011,  // (5 - sqrt(5))/20  G4
                        0.276393202250021,  // 2 * G4
                        0.414589803375032,  // 3 * G4
                       -0.447213595499958); // -1 + 4 * G4

// First corner
  vec4 i  = floor(v + dot(v, vec4(F4)) );
  vec4 x0 = v -   i + dot(i, C.xxxx);

// Other corners

// Rank sorting originally contributed by Bill Licea-Kane, AMD (formerly ATI)
  vec4 i0;
  vec3 isX = step( x0.yzw, x0.xxx );
  vec3 isYZ = step( x0.zww, x0.yyz );
//  i0.x = dot( isX, vec3( 1.0 ) );
  i0.x = isX.x + isX.y + isX.z;
  i0.yzw = 1.0 - isX;
//  i0.y += dot( isYZ.xy, vec2( 1.0 ) );
  i0.y += isYZ.x + isYZ.y;
  i0.zw += 1.0 - isYZ.xy;
  i0.z += isYZ.z;
  i0.w += 1.0 - isYZ.z;

  // i0 now contains the unique values 0,1,2,3 in each channel
  vec4 i3 = clamp( i0, 0.0, 1.0 );
  vec4 i2 = clamp( i0-1.0, 0.0, 1.0 );
  vec4 i1 = clamp( i0-2.0, 0.0, 1.0 );

  //  x0 = x0 - 0.0 + 0.0 * C.xxxx
  //  x1 = x0 - i1  + 1.0 * C.xxxx
  //  x2 = x0 - i2  + 2.0 * C.xxxx
  //  x3 = x0 - i3  + 3.0 * C.xxxx
  //  x4 = x0 - 1.0 + 4.0 * C.xxxx
  vec4 x1 = x0 - i1 + C.xxxx;
  vec4 x2 = x0 - i2 + C.yyyy;
  vec4 x3 = x0 - i3 + C.zzzz;
  vec4 x4 = x0 + C.wwww;

// Permutations
  i = mod289(i); 
  float j0 = permute( permute( permute( permute(i.w) + i.z) + i.y) + i.x);
  vec4 j1 = permute( permute( permute( permute (
             i.w + vec4(i1.w, i2.w, i3.w, 1.0 ))
           + i.z + vec4(i1.z, i2.z, i3.z, 1.0 ))
           + i.y + vec4(i1.y, i2.y, i3.y, 1.0 ))
           + i.x + vec4(i1.x, i2.x, i3.x, 1.0 ));

// Gradients: 7x7x6 points over a cube, mapped onto a 4-cross polytope
// 7*7*6 = 294, which is close to the ring size 17*17 = 289.
  vec4 ip = vec4(1.0/294.0, 1.0/49.0, 1.0/7.0, 0.0) ;

  vec4 p0 = grad4(j0,   ip);
  vec4 p1 = grad4(j1.x, ip);
  vec4 p2 = grad4(j1.y, ip);
  vec4 p3 = grad4(j1.z, ip);
  vec4 p4 = grad4(j1.w, ip);

// Normalise gradients
  vec4 norm = taylorInvSqrt(vec4(dot(p0,p0), dot(p1,p1), dot(p2, p2), dot(p3,p3)));
  p0 *= norm.x;
  p1 *= norm.y;
  p2 *= norm.z;
  p3 *= norm.w;
  p4 *= taylorInvSqrt(dot(p4,p4));

// Mix contributions from the five corners
  vec3 m0 = max(0.6 - vec3(dot(x0,x0), dot(x1,x1), dot(x2,x2)), 0.0);
  vec2 m1 = max(0.6 - vec2(dot(x3,x3), dot(x4,x4)            ), 0.0);
  m0 = m0 * m0;
  m1 = m1 * m1;
  return 49.0 * ( dot(m0*m0, vec3( dot( p0, x0 ), dot( p1, x1 ), dot( p2, x2 )))
               + dot(m1*m1, vec2( dot( p3, x3 ), dot( p4, x4 ) ) ) ) ;

  }

  
//-------------------------------------------------------------------------------
```

</details>

### Anexo B — Como o compilador gera os shaders

O utilizador não escreve GLSL, mas os shaders continuam a existir: é o compilador que os escreve. Para cada material, o compilador monta um par `.vert` / `.frag` a partir de **blocos de código pré-definidos**, escolhidos pelo modo de iluminação e pelas camadas. Os blocos são os próprios shaders das aulas 2 e 3, partidos em pedaços reutilizáveis.

**Correspondência entre a Caravela e o GLSL gerado:**

| Na Caravela | O que o compilador põe nos shaders | Origem nas aulas |
|---|---|---|
| (sempre) | vertex shader base: transforma a posição (`m_pvm * position`) e passa a normal e a direção da luz para o espaço da câmara | todos |
| `lighting none` | fragment: a cor final é a cor base, sem luz | — |
| `lighting per_vertex` | vertex: calcula `intensity = max(0, dot(n, l))` e a cor `intensity * diffuse`; fragment: só copia a cor recebida | `simple.vert` / `simple.frag` (aula 2) |
| `lighting phong { shininess s; }` | vertex: passa `n`, `l` e o vetor para o olho `e`; fragment: calcula a intensidade em cada píxel e soma o brilho especular `pow(max(0, dot(h, n)), s)`, com `h` o vetor a meio entre `l` e `e` | `v4.vert` / `v4.frag` (aula 2) |
| `lighting toon { levels k; }` | como `per_vertex`, mas arredonda `intensity` para `k` patamares | variante do `v4.vert` com 4 patamares (aula 2) |
| `layer day t` | vertex: passa as coordenadas de textura; fragment: a cor base passa a ser `texture(t, uv)` em vez de `diffuse` | `texture.vert` / `texture.frag` (aula 3) |
| `layer specular t` | fragment: o brilho especular é multiplicado por `texture(t, uv).r` (só brilha onde a imagem é clara) | `texture.frag` (aula 3) |
| `layer night t` | fragment: `mix(cor_noite, cor_dia, smoothstep(0.0, 0.1, intensity))`, para as luzes da cidade aparecerem só no lado escuro | `texture.frag` (aula 3) |
| `layer clouds noise speed v` | junta o ficheiro de ruído ao fragment shader; fragment: calcula nuvens com `perlinNoise(..., time * v)` e mistura-as com branco | `noise.frag` + `texture.frag` (aula 3) |

**Um shader por combinação, não por material.** O shader só depende da *forma* do material: o modo de iluminação e as camadas que tem. Não depende dos *valores*: a cor, a `shininess`, a direção da luz ou a posição da câmara não ficam escritas no GLSL. Ficam no `.mlib` e o Nau entrega-as ao shader através de uniforms, tal como já acontece nas aulas. Por isso, um material amarelo e um azul, ambos `phong`, usam o **mesmo** `phong.frag`, e mudar a luz de sítio não gera shader nenhum. O número de shaders diferentes é pequeno e limitado: no máximo um por cada combinação de modo e camadas que apareça no programa.

**O que é gerado.** Para cada combinação usada, o compilador escreve um par de shaders com um nome derivado dela (`phong.vert`/`phong.frag`, `phong_day_specular_night_clouds.frag`, ...). Os blocos são concatenados pela ordem da tabela, e cada bloco declara os uniforms de que precisa. O compilador junta essas declarações, usando sempre os mesmos nomes (`m_pvm`, `m_view`, `m_normal`, `l_dir`, `diffuse`, `shininess`, `time`, `tex0`, `tex1`, ...). Para cada uniform escreve no `.mlib` a linha `<valueof>` correspondente, e para cada camada uma unidade de textura (0, 1, 2, ...). Como os dois lados (o shader e o `.mlib`) saem do mesmo sítio, os nomes coincidem sempre.

Por exemplo, para o material

```
material amarelo {
    diffuse rgb(1.0, 0.87, 0.0);
    lighting phong { shininess 100; }
}
```

o compilador vê a combinação "`phong`, sem camadas" e gera (uma única vez) um `phong.frag` equivalente ao `v4.frag` da aula 2:

```glsl
#version 460
in vec3 n, l, e;
uniform vec4  diffuse;
uniform float shininess;
out vec4 color;

void main() {
    vec3 nn = normalize(n);
    float intensity = max(0.0, dot(nn, normalize(l)));
    vec4 specular = vec4(0.0);
    if (intensity > 0.0) {
        vec3 h = normalize(normalize(e) + normalize(l));
        specular = vec4(1.0) * pow(max(0.0, dot(h, nn)), shininess);
    }
    color = max(0.25, intensity) * diffuse + specular;
}
```

A cor e o brilho do `amarelo` vão para o `.mlib`, que diz ao Nau para os pôr nos uniforms `diffuse` e `shininess`:

```xml
<material name="amarelo">
    <color>
        <DIFFUSE r="1.0" g="0.87" b="0.0" a="1.0" />
        <SHININESS value="100" />
    </color>
    <shader name="phong">
        <values>
            <valueof uniform="m_pvm"     type="RENDERER" component="PROJECTION_VIEW_MODEL" />
            <valueof uniform="m_view"    type="RENDERER" component="VIEW" />
            <valueof uniform="m_normal"  type="RENDERER" component="NORMAL" />
            <valueof uniform="l_dir"     type="LIGHT"    component="DIRECTION" id="0" />
            <valueof uniform="diffuse"   type="COLOR"    component="DIFFUSE" />
            <valueof uniform="shininess" type="COLOR"    component="SHININESS" />
        </values>
    </shader>
</material>
```

Um segundo material `azul` com `lighting phong` gera outro bloco `<material>` igual a este, só com outros números em `<DIFFUSE>`, e aponta para o mesmo `<shader name="phong">`. Este é o mesmo mecanismo que o `simple.mlib` da aula 2 já usa. A direção da luz também nunca entra no shader: vem da declaração `light` e é entregue em `l_dir`.

**Ligação com a gramática de atributos.** O conjunto de blocos de cada material, e por isso o conjunto de uniforms e de unidades de textura, pode ser descrito como um **atributo sintetizado**. Sobe das declarações `lighting` e `layer` até ao nó `material`, e é a partir dele que se gera o `.mlib`. As regras de coerência da secção 1.5 (por exemplo, `layer specular` só com `phong`) garantem que os blocos escolhidos encaixam uns nos outros.